In [1]:
import pandas as pd
import numpy as np

In [2]:
import pandas as pd

df = pd.read_csv("safety_incidents_synthetic_raw.csv")
df.head()

,incident_id,incident_date,incident_time,department,location,shift,employee_role,experience_level,equipment_type,equipment_status,...,severity_level,risk_level,risk_score,immediate_action,corrective_action,preventive_action,reported_by_role,investigation_status,incident_status,lost_time_days
0,INC-2021-00001,1/1/2021,0:09,Chemical Processing,Effluent Treatment Plant,Night,Process Technician,1-3 years,Chemical Storage Cabinet,Damaged,...,Serious,High,61.9,"Employee moved to fresh air, contaminated clot...",Review the chemical transfer and handling meth...,"Set up PPE compliance audits, stock availabili...",Shift Manager,Completed,Closed,8
1,INC-2021-00002,1/1/2021,5:10,Assembly,Assembly Line B,Night,Line Supervisor,3-7 years,Pneumatic Tool,Operational,...,Serious,High,52.6,Machine stopped and isolated; injured employee...,"Inspect the air-powered driver, repair or repl...","Implement permit-to-work audits, lock-out/tag-...",Supervisor,Completed,Closed,17
2,INC-2021-00003,1/1/2021,7:25,Production,Machining Bay,Morning,Production Technician,More than 7 years,Forklift,Operational,...,Minor,Medium,27.6,Area secured and injured employee sent to the ...,"Secure loose material, install a barrier aroun...",Standardise the shift handover format and use ...,Supervisor,Completed,Closed,0
3,INC-2021-00004,1/1/2021,8:52,Quality Control & Laboratory,Incoming Inspection,Morning,Quality Inspector,3-7 years,Fume Hood,Operational,...,Minor,Medium,27.8,"Employee moved to fresh air, contaminated clot...",Review the chemical transfer and handling meth...,Standardise the shift handover format and use ...,Supervisor,Completed,Closed,0
4,INC-2021-00005,1/1/2021,9:26,Assembly,Sub-assembly Area,Morning,Assembly Operator,More than 7 years,Not Applicable,Not Applicable,...,Moderate,Low,19.3,Task rotated and injured employee sent for med...,"Adjust the workstation layout, provide adjusta...",Include HSE review at the design and modificat...,Co-worker,Completed,Closed,1


In [3]:
df.columns

Index(['incident_id', 'incident_date', 'incident_time', 'department',
       'location', 'shift', 'employee_role', 'experience_level',
       'equipment_type', 'equipment_status', 'incident_type', 'hazard_type',
       'incident_description', 'injury_type', 'body_part_affected',
       'medical_treatment_required', 'root_cause_category',
       'root_cause_description', 'contributing_factor_1',
       'contributing_factor_2', 'severity_level', 'risk_level', 'risk_score',
       'immediate_action', 'corrective_action', 'preventive_action',
       'reported_by_role', 'investigation_status', 'incident_status',
       'lost_time_days'],
      dtype='str')

In [4]:
file_path = "safety_incidents_synthetic_raw.csv"

In [5]:
raw_df = pd.read_csv(
    file_path
)

df = raw_df.copy()

print("Raw shape:", raw_df.shape)
print("Working shape:", df.shape)

Raw shape: (8000, 30)
Working shape: (8000, 30)


In [6]:
# Standardize column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# Strip leading/trailing whitespace from all text columns
text_columns = df.select_dtypes(include="object").columns

for col in text_columns:
    df[col] = df[col].str.strip()

# Convert empty strings / whitespace-only strings to NaN
df[text_columns] = df[text_columns].replace(
    r"^\s*$",
    np.nan,
    regex=True
)

print("Cleaning of text formatting completed.")

Cleaning of text formatting completed.


C:\Users\Kartik Singh\AppData\Local\Temp\ipykernel_13060\2544212027.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = df.select_dtypes(include="object").columns


In [7]:
# Convert incident date
df["incident_date"] = pd.to_datetime(
    df["incident_date"],
    errors="coerce"
)

print(df["incident_time"].head(20).to_list())
print("\nData type:", df["incident_time"].dtype)
print("\nUnique examples:")
print(df["incident_time"].dropna().astype(str).head(30).to_list())

print("Invalid dates:", df["incident_date"].isna().sum())


df["incident_time"] = pd.to_datetime(
    df["incident_time"],
    format="%H:%M",
    errors="coerce"
).dt.time

print("Invalid times:", df["incident_time"].isna().sum())
print(df["incident_time"].head(20).to_list())

['0:09', '5:10', '7:25', '8:52', '9:26', '12:19', '13:39', '15:49', '7:12', '19:18', '16:40', '4:52', '6:33', '7:23', '8:59', '15:52', '1:57', '5:39', '6:04', '6:34']

Data type: str

Unique examples:
['0:09', '5:10', '7:25', '8:52', '9:26', '12:19', '13:39', '15:49', '7:12', '19:18', '16:40', '4:52', '6:33', '7:23', '8:59', '15:52', '1:57', '5:39', '6:04', '6:34', '6:51', '10:11', '15:12', '16:14', '20:15', '4:11', '10:48', '17:40', '0:10', '1:09']
Invalid dates: 0
Invalid times: 0
[datetime.time(0, 9), datetime.time(5, 10), datetime.time(7, 25), datetime.time(8, 52), datetime.time(9, 26), datetime.time(12, 19), datetime.time(13, 39), datetime.time(15, 49), datetime.time(7, 12), datetime.time(19, 18), datetime.time(16, 40), datetime.time(4, 52), datetime.time(6, 33), datetime.time(7, 23), datetime.time(8, 59), datetime.time(15, 52), datetime.time(1, 57), datetime.time(5, 39), datetime.time(6, 4), datetime.time(6, 34)]


In [8]:
print("Shape:", df.shape)

print("\nMissing values:")
print(
    df.isna()
      .sum()
      .sort_values(ascending=False)
)

print("\nDuplicate rows:", df.duplicated().sum())

print("\nData types:")
print(df.dtypes)

Shape: (8000, 30)

Missing values:
contributing_factor_2         2578
injury_type                   1135
body_part_affected            1135
experience_level               148
reported_by_role                87
equipment_status                63
employee_role                   50
root_cause_description          41
department                       0
incident_time                    0
shift                            0
location                         0
incident_id                      0
incident_date                    0
incident_description             0
hazard_type                      0
incident_type                    0
equipment_type                   0
root_cause_category              0
contributing_factor_1            0
severity_level                   0
medical_treatment_required       0
risk_level                       0
risk_score                       0
corrective_action                0
immediate_action                 0
preventive_action                0
investigation_status

In [9]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": (df.isna().mean() * 100).round(2)
})

missing_summary = missing_summary[
    missing_summary["missing_count"] > 0
].sort_values("missing_count", ascending=False)

display(missing_summary)

,missing_count,missing_percentage
contributing_factor_2,2578,32.22
injury_type,1135,14.19
body_part_affected,1135,14.19
experience_level,148,1.85
reported_by_role,87,1.09
equipment_status,63,0.79
employee_role,50,0.62
root_cause_description,41,0.51


In [10]:
for col in missing_summary.index:
    print(f"\n{'='*70}")
    print(f"{col}")
    print(f"{'='*70}")
    print(df.loc[df[col].isna()].head(5))


contributing_factor_2
       incident_id incident_date incident_time             department  \
12  INC-2021-00013    2021-01-04      06:33:00             Production   
14  INC-2021-00015    2021-01-04      08:59:00  Fabrication & Welding   
15  INC-2021-00016    2021-01-04      15:52:00    Chemical Processing   
16  INC-2021-00017    2021-01-05      01:57:00             Production   
24  INC-2021-00025    2021-01-05      20:15:00               Assembly   

                    location    shift          employee_role  \
12                Press Shop  Morning  Production Technician   
14          Paint Booth Area  Morning             Fabricator   
15  Effluent Treatment Plant  Evening      Chemical Operator   
16         Production Line 2    Night                    NaN   
24           Assembly Line A  Evening      Production Helper   

     experience_level              equipment_type equipment_status  ...  \
12  More than 7 years                   CNC Lathe   Malfunctioning  ...   
14 

In [11]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": (df.isna().mean() * 100).round(2),
    "non_missing_count": df.notna().sum()
})

missing_summary = (
    missing_summary[missing_summary["missing_count"] > 0]
    .sort_values("missing_count", ascending=False)
)

display(missing_summary)

,missing_count,missing_percentage,non_missing_count
contributing_factor_2,2578,32.22,5422
injury_type,1135,14.19,6865
body_part_affected,1135,14.19,6865
experience_level,148,1.85,7852
reported_by_role,87,1.09,7913
equipment_status,63,0.79,7937
employee_role,50,0.62,7950
root_cause_description,41,0.51,7959


In [12]:
target_cols = [
    "root_cause_category",
    "severity_level",
    "risk_level"
]

for target in target_cols:
    print(f"\n{'='*80}")
    print(f"Missing values vs {target}")
    print(f"{'='*80}")

    for col in missing_summary.index:
        if col != target:
            missing_count = df.loc[df[col].isna(), target].value_counts()
            print(f"\n{col}:")
            print(missing_count)


Missing values vs root_cause_category

contributing_factor_2:
root_cause_category
Human Error                   288
Procedure Violation           275
Poor Housekeeping             259
Unsafe Working Condition      223
Inadequate Training           211
Fatigue/Workload              194
Inadequate Risk Assessment    153
Poor Maintenance              150
Supervision Failure           140
Design Deficiency             135
Communication Failure         125
Equipment Failure             123
Environmental Condition       117
PPE Non-Compliance            115
Other                          70
Name: count, dtype: int64

injury_type:
root_cause_category
Human Error                   193
Procedure Violation           146
Unsafe Working Condition      118
Communication Failure         102
Poor Maintenance               92
Inadequate Training            88
Inadequate Risk Assessment     74
Equipment Failure              71
Poor Housekeeping              69
Supervision Failure            65
Design 

In [13]:
# Fill small categorical missing values
fill_unknown = [
    "experience_level",
    "reported_by_role",
    "equipment_status",
    "employee_role"
]

for col in fill_unknown:
    df[col] = df[col].fillna("Unknown")

print("Missing-value treatment completed.")

Missing-value treatment completed.


In [14]:
missing_after = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": (df.isna().mean() * 100).round(2)
})

missing_after = missing_after[
    missing_after["missing_count"] > 0
].sort_values("missing_count", ascending=False)

display(missing_after)

,missing_count,missing_percentage
contributing_factor_2,2578,32.22
injury_type,1135,14.19
body_part_affected,1135,14.19
root_cause_description,41,0.51


In [15]:
# Check duplicate rows
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


In [16]:
# Check duplicate incident IDs
duplicate_ids = df["incident_id"].duplicated().sum()

print("Duplicate incident IDs:", duplicate_ids)

Duplicate incident IDs: 0


In [17]:
categorical_cols = df.select_dtypes(include="object").columns

for col in categorical_cols:
    print(f"\n{'='*50}")
    print(f"{col}")
    print(f"{'='*50}")
    print(df[col].value_counts(dropna=False))


incident_id
incident_id
INC-2021-00001    1
INC-2021-00002    1
INC-2021-00003    1
INC-2021-00004    1
INC-2021-00005    1
                 ..
INC-2026-00892    1
INC-2026-00893    1
INC-2026-00894    1
INC-2026-00895    1
INC-2026-00896    1
Name: count, Length: 8000, dtype: int64

incident_time
incident_time
07:26:00    17
08:40:00    17
08:59:00    16
12:50:00    16
11:36:00    16
            ..
02:41:00     1
21:00:00     1
03:20:00     1
21:03:00     1
02:28:00     1
Name: count, Length: 1415, dtype: int64

department
department
Production                      1730
Warehouse & Logistics           1205
Maintenance                     1069
Assembly                         776
Fabrication & Welding            709
Chemical Processing              637
Electrical                       548
Packaging                        530
Utilities                        414
Quality Control & Laboratory     220
Facilities & Housekeeping        162
Name: count, dtype: int64

location
location
Plant 

C:\Users\Kartik Singh\AppData\Local\Temp\ipykernel_13060\3856595372.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include="object").columns


In [18]:
numeric_cols = df.select_dtypes(include="number").columns

print("Numeric columns:")
print(list(numeric_cols))

for col in numeric_cols:
    print(f"\n{col}")
    print(df[col].describe())
    print("Negative values:", (df[col] < 0).sum())

Numeric columns:
['risk_score', 'lost_time_days']

risk_score
count    8000.000000
mean       41.610238
std        20.654510
min         1.900000
25%        25.000000
50%        39.500000
75%        56.300000
max       100.000000
Name: risk_score, dtype: float64
Negative values: 0

lost_time_days
count    8000.000000
mean       14.529500
std        42.526712
min         0.000000
25%         0.000000
50%         1.000000
75%         9.000000
max       365.000000
Name: lost_time_days, dtype: float64
Negative values: 0


In [19]:
# Date validation
print("Incident date:")
print("Min:", df["incident_date"].min())
print("Max:", df["incident_date"].max())
print("Missing:", df["incident_date"].isna().sum())

# Time validation
print("\nIncident time:")
print("Missing:", df["incident_time"].isna().sum())
print("Unique times:", df["incident_time"].nunique())

Incident date:
Min: 2021-01-01 00:00:00
Max: 2026-08-31 00:00:00
Missing: 0

Incident time:
Missing: 0
Unique times: 1415


In [20]:
print(
    "Future incident dates:",
    (df["incident_date"] > pd.Timestamp.today()).sum()
)

Future incident dates: 0


In [21]:
# Outlier check using IQR

for col in ["risk_score", "lost_time_days"]:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = ((df[col] < lower) | (df[col] > upper)).sum()

    print(f"\n{col}")
    print("Q1:", Q1)
    print("Q3:", Q3)
    print("IQR:", IQR)
    print("Lower bound:", lower)
    print("Upper bound:", upper)
    print("Potential outliers:", outliers)


risk_score
Q1: 25.0
Q3: 56.3
IQR: 31.299999999999997
Lower bound: -21.949999999999996
Upper bound: 103.25
Potential outliers: 0

lost_time_days
Q1: 0.0
Q3: 9.0
IQR: 9.0
Lower bound: -13.5
Upper bound: 22.5
Potential outliers: 1058


In [22]:
df = df.drop(columns=["incident_id"])

print("Remaining columns:", len(df.columns))
print(df.columns.tolist())

Remaining columns: 29
['incident_date', 'incident_time', 'department', 'location', 'shift', 'employee_role', 'experience_level', 'equipment_type', 'equipment_status', 'incident_type', 'hazard_type', 'incident_description', 'injury_type', 'body_part_affected', 'medical_treatment_required', 'root_cause_category', 'root_cause_description', 'contributing_factor_1', 'contributing_factor_2', 'severity_level', 'risk_level', 'risk_score', 'immediate_action', 'corrective_action', 'preventive_action', 'reported_by_role', 'investigation_status', 'incident_status', 'lost_time_days']


In [23]:
print("Number of columns:", len(df.columns))

for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")

Number of columns: 29
1. incident_date
2. incident_time
3. department
4. location
5. shift
6. employee_role
7. experience_level
8. equipment_type
9. equipment_status
10. incident_type
11. hazard_type
12. incident_description
13. injury_type
14. body_part_affected
15. medical_treatment_required
16. root_cause_category
17. root_cause_description
18. contributing_factor_1
19. contributing_factor_2
20. severity_level
21. risk_level
22. risk_score
23. immediate_action
24. corrective_action
25. preventive_action
26. reported_by_role
27. investigation_status
28. incident_status
29. lost_time_days


In [24]:
# Main input features
feature_cols = [
    "incident_date",
    "incident_time",
    "department",
    "location",
    "shift",
    "employee_role",
    "experience_level",
    "equipment_type",
    "equipment_status",
    "incident_type",
    "hazard_type",
    "incident_description",
    "contributing_factor_1",
    "contributing_factor_2",
    "injury_type",
    "body_part_affected",
    "medical_treatment_required",
    "reported_by_role"
]

# Targets
target_root_cause = "root_cause_category"
target_severity = "severity_level"
target_risk = "risk_level"

In [25]:
# Root Cause Prediction
X_root_cause = df[feature_cols].copy()
y_root_cause = df[target_root_cause].copy()

# Severity Prediction
X_severity = df[feature_cols].copy()
y_severity = df[target_severity].copy()

# Risk Level Prediction
X_risk = df[feature_cols].copy()
y_risk = df[target_risk].copy()

In [26]:
print("ROOT CAUSE MODEL")
print("X:", X_root_cause.shape)
print("y:", y_root_cause.shape)

print("\nSEVERITY MODEL")
print("X:", X_severity.shape)
print("y:", y_severity.shape)

print("\nRISK MODEL")
print("X:", X_risk.shape)
print("y:", y_risk.shape)

ROOT CAUSE MODEL
X: (8000, 18)
y: (8000,)

SEVERITY MODEL
X: (8000, 18)
y: (8000,)

RISK MODEL
X: (8000, 18)
y: (8000,)


In [27]:
# Target class distributions

print("ROOT CAUSE DISTRIBUTION")
print(y_root_cause.value_counts())
print("\nPercentage:")
print((y_root_cause.value_counts(normalize=True) * 100).round(2))

print("\n" + "="*60)

print("SEVERITY DISTRIBUTION")
print(y_severity.value_counts())
print("\nPercentage:")
print((y_severity.value_counts(normalize=True) * 100).round(2))

print("\n" + "="*60)

print("RISK LEVEL DISTRIBUTION")
print(y_risk.value_counts())
print("\nPercentage:")
print((y_risk.value_counts(normalize=True) * 100).round(2))

ROOT CAUSE DISTRIBUTION
root_cause_category
Procedure Violation           905
Human Error                   894
Poor Housekeeping             787
Unsafe Working Condition      694
Inadequate Training           669
Fatigue/Workload              567
Poor Maintenance              495
Inadequate Risk Assessment    487
Supervision Failure           451
Design Deficiency             413
Equipment Failure             384
Communication Failure         359
PPE Non-Compliance            352
Environmental Condition       348
Other                         195
Name: count, dtype: int64

Percentage:
root_cause_category
Procedure Violation           11.31
Human Error                   11.18
Poor Housekeeping              9.84
Unsafe Working Condition       8.67
Inadequate Training            8.36
Fatigue/Workload               7.09
Poor Maintenance               6.19
Inadequate Risk Assessment     6.09
Supervision Failure            5.64
Design Deficiency              5.16
Equipment Failure          

In [28]:
categorical_features = [
    "department",
    "location",
    "shift",
    "employee_role",
    "experience_level",
    "equipment_type",
    "equipment_status",
    "incident_type",
    "hazard_type",
    "injury_type",
    "body_part_affected",
    "medical_treatment_required",
    "reported_by_role"
]

for col in categorical_features:
    print(f"\n{'='*60}")
    print(col)
    print(f"{'='*60}")
    print(df[col].value_counts(dropna=False))


department
department
Production                      1730
Warehouse & Logistics           1205
Maintenance                     1069
Assembly                         776
Fabrication & Welding            709
Chemical Processing              637
Electrical                       548
Packaging                        530
Utilities                        414
Quality Control & Laboratory     220
Facilities & Housekeeping        162
Name: count, dtype: int64

location
location
Plant Floor                 326
Machining Bay               300
Production Line 2           300
Press Shop                  292
Stamping Area               291
Production Line 1           280
Moulding Area               267
Maintenance Workshop        241
Yard                        220
Assembly Line B             207
Utility Yard                207
Motor Repair Bay            207
Tool Crib                   206
Final Inspection Bay        204
Despatch Area               203
Finished Goods Store        198
Main Warehous

In [29]:
import pandas as pd

for target in ["root_cause_category", "severity_level", "risk_level"]:
    print(f"\n{'='*70}")
    print(f"INCIDENT TYPE vs {target}")
    print(f"{'='*70}")

    table = pd.crosstab(
        df["incident_type"],
        df[target],
        normalize="index"
    ) * 100

    display(table.round(2))


INCIDENT TYPE vs root_cause_category


root_cause_category,Communication Failure,Design Deficiency,Environmental Condition,Equipment Failure,Fatigue/Workload,Human Error,Inadequate Risk Assessment,Inadequate Training,Other,PPE Non-Compliance,Poor Housekeeping,Poor Maintenance,Procedure Violation,Supervision Failure,Unsafe Working Condition
incident_type,,,,,,,,,,,,,,,
Caught In/Between,2.77,15.51,0.00,9.70,5.26,11.08,4.71,10.25,0.00,0.00,0.00,6.65,26.04,8.03,0.00
Chemical Exposure,4.70,0.00,0.00,1.88,0.63,10.66,8.46,11.91,0.00,31.35,5.33,0.00,15.99,5.02,4.08
Chemical Spill,3.41,7.39,0.00,6.82,0.57,23.30,10.23,12.50,0.00,0.00,3.41,10.80,15.34,2.27,3.98
Electrical Burn,6.03,3.45,0.00,10.34,0.86,6.90,12.07,5.17,0.00,18.10,0.00,18.97,17.24,0.86,0.00
Electrical Shock,11.44,7.01,0.00,8.86,0.37,10.70,7.01,6.64,0.00,5.17,0.00,9.23,26.57,0.37,6.64
Ergonomic Injury,0.00,21.45,0.00,0.00,29.60,5.59,12.59,9.79,5.36,0.00,0.00,0.00,0.00,7.23,8.39
Explosion,8.64,14.81,1.23,13.58,2.47,2.47,14.81,8.64,1.23,0.00,0.00,14.81,11.11,0.00,6.17
Falling Object,4.28,0.00,0.00,5.23,0.48,11.40,6.41,5.70,0.00,0.00,17.34,4.28,15.20,5.94,23.75
Fire,0.00,4.66,2.85,10.88,0.52,6.99,6.48,3.89,2.33,0.00,13.99,17.10,18.39,0.00,11.92



INCIDENT TYPE vs severity_level


severity_level,Critical,Minor,Moderate,Serious,Severe
incident_type,,,,,
Caught In/Between,3.88,14.13,24.10,30.47,27.42
Chemical Exposure,0.00,42.32,41.69,14.42,1.57
Chemical Spill,1.14,32.95,47.73,15.91,2.27
Electrical Burn,5.17,5.17,20.69,49.14,19.83
Electrical Shock,2.58,25.09,39.11,26.57,6.64
Ergonomic Injury,0.00,33.57,52.91,12.12,1.40
Explosion,27.16,0.00,8.64,24.69,39.51
Falling Object,4.99,21.62,30.40,24.47,18.53
Fire,5.44,9.59,35.49,33.42,16.06



INCIDENT TYPE vs risk_level


risk_level,Critical,High,Low,Medium
incident_type,,,,
Caught In/Between,23.82,48.48,1.94,25.76
Chemical Exposure,0.63,24.76,15.99,58.62
Chemical Spill,3.98,25.57,9.66,60.80
Electrical Burn,31.03,53.45,0.00,15.52
Electrical Shock,11.07,49.45,0.74,38.75
Ergonomic Injury,0.00,3.26,55.01,41.72
Explosion,62.96,29.63,0.00,7.41
Falling Object,14.25,35.15,9.03,41.57
Fire,15.54,43.01,7.77,33.68


In [30]:
print(
    df.groupby("severity_level")["lost_time_days"]
      .agg(["count", "mean", "median", "min", "max"])
      .round(2)
)

                count    mean  median  min  max
severity_level                                 
Critical          163  241.38   239.0    0  365
Minor            2721    0.11     0.0    0    1
Moderate         2700    1.89     2.0    0    4
Serious          1639   12.95    13.0    0   25
Severe            777   64.69    64.0    0  120


In [31]:
print(
    df.groupby("risk_level")["lost_time_days"]
      .agg(["count", "mean", "median", "min", "max"])
      .round(2)
)

            count   mean  median  min  max
risk_level                                
Critical      559  97.58    63.0    0  365
High         2109  22.17     9.0    0  365
Low          2009   0.76     0.0    0   25
Medium       3323   4.03     1.0    0  119


In [32]:
# Convert incident_time back to pandas datetime
df["incident_time"] = pd.to_datetime(
    df["incident_time"].astype(str),
    format="%H:%M:%S",
    errors="coerce"
)

print(df["incident_time"].dtype)
print("Missing times:", df["incident_time"].isna().sum())

datetime64[us]
Missing times: 0


In [33]:
df["incident_year"] = df["incident_date"].dt.year
df["incident_month"] = df["incident_date"].dt.month
df["incident_hour"] = df["incident_time"].dt.hour

print("Incidents by year:")
print(df["incident_year"].value_counts().sort_index())

print("\nIncidents by month:")
print(df["incident_month"].value_counts().sort_index())

print("\nIncidents by hour:")
print(df["incident_hour"].value_counts().sort_index())

Incidents by year:
incident_year
2021    1481
2022    1413
2023    1398
2024    1481
2025    1331
2026     896
Name: count, dtype: int64

Incidents by month:
incident_month
1     698
2     692
3     711
4     727
5     662
6     690
7     735
8     755
9     559
10    628
11    561
12    582
Name: count, dtype: int64

Incidents by hour:
incident_hour
0     241
1     232
2     240
3     273
4     221
5     171
6     327
7     504
8     577
9     586
10    535
11    471
12    390
13    286
14    276
15    366
16    434
17    380
18    385
19    340
20    255
21    172
22    146
23    192
Name: count, dtype: int64


In [34]:
print("\nSeverity by shift:")
display(
    pd.crosstab(
        df["shift"],
        df["severity_level"],
        normalize="index"
    ).mul(100).round(2)
)

print("\nRisk by shift:")
display(
    pd.crosstab(
        df["shift"],
        df["risk_level"],
        normalize="index"
    ).mul(100).round(2)
)


Severity by shift:


severity_level,Critical,Minor,Moderate,Serious,Severe
shift,,,,,
Evening,2.15,34.32,33.47,20.63,9.43
Morning,2.01,33.08,34.39,20.78,9.74
Night,1.92,35.55,32.81,19.64,10.08



Risk by shift:


risk_level,Critical,High,Low,Medium
shift,,,,
Evening,7.13,26.07,24.16,42.64
Morning,7.10,26.61,24.97,41.32
Night,6.53,26.28,26.86,40.33


In [35]:
# NLP basic statistics

df["description_word_count"] = (
    df["incident_description"]
    .str.split()
    .str.len()
)

df["description_char_count"] = (
    df["incident_description"]
    .str.len()
)

print("Word count statistics:")
print(df["description_word_count"].describe())

print("\nCharacter count statistics:")
print(df["description_char_count"].describe())

print("\nEmpty descriptions:")
print((df["incident_description"].str.strip() == "").sum())

print("\nMissing descriptions:")
print(df["incident_description"].isna().sum())

Word count statistics:
count    8000.000000
mean       49.794000
std        13.517143
min        11.000000
25%        42.000000
50%        51.000000
75%        59.000000
max       102.000000
Name: description_word_count, dtype: float64

Character count statistics:
count    8000.000000
mean      283.557750
std        76.516256
min        61.000000
25%       242.000000
50%       290.000000
75%       335.250000
max       560.000000
Name: description_char_count, dtype: float64

Empty descriptions:
0

Missing descriptions:
0


In [36]:
print("\nShortest descriptions:")
display(
    df[["incident_description", "description_word_count"]]
    .sort_values("description_word_count")
    .head(10)
)

print("\nLongest descriptions:")
display(
    df[["incident_description", "description_word_count"]]
    .sort_values("description_word_count", ascending=False)
    .head(10)
)


Shortest descriptions:


,incident_description,description_word_count
7193,A quality inspector spilled solvent while deca...,11
2436,A forklift truck struck a driver while reversi...,11
3825,A worker received an electric shock while work...,11
6474,A pallet jack struck the forklift operator whi...,12
7270,An employee suffered irritation after handling...,12
6051,A flange joint on the gas cylinder manifold re...,12
1033,A steel bracket fell from the overhead rack an...,12
312,The press cycled unexpectedly while a CNC oper...,12
456,A forklift struck a production helper while re...,13
6550,An electrician tripped on a raised expansion j...,13



Longest descriptions:


,incident_description,description_word_count
5007,"In the Paint Booth Area, a welder was burned w...",102
6120,"During the evening shift, a wrench dropped fro...",89
4351,"During the evening shift, a hydraulic hose on ...",89
146,"While setting up the chemical dosing pump, the...",89
4688,"During the evening shift, a production helper ...",88
2522,"At around 7:48 am, a fitter tripped on a loose...",86
4979,"During the evening shift, a production helper ...",86
752,"While the night shift was in progress, a forkl...",86
1276,"At around 11:36 am, a bolt fell from a worker'...",86
4023,"In the Process Control Area, a gas accumulatio...",85


In [37]:
from collections import Counter
import re

# Combine all incident descriptions
text = " ".join(df["incident_description"].astype(str))

# Basic tokenization
words = re.findall(r"\b[a-zA-Z]+\b", text.lower())

word_counts = Counter(words)

print("Total words:", len(words))
print("Unique words:", len(word_counts))

print("\nTop 30 most frequent words:")
for word, count in word_counts.most_common(30):
    print(f"{word}: {count}")

Total words: 398188
Unique words: 1380

Top 30 most frequent words:
the: 35685
a: 19390
was: 17897
and: 14115
to: 10009
for: 7771
in: 6923
he: 6375
of: 5938
on: 4563
had: 4103
while: 3715
his: 3413
not: 3146
she: 3092
treatment: 2915
at: 2893
shift: 2732
an: 2707
been: 2679
area: 2553
injury: 2506
no: 2369
from: 2321
work: 2256
with: 2218
operator: 1963
first: 1841
aid: 1833
duty: 1823


In [38]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=1000
)

X_text = tfidf.fit_transform(df["incident_description"])

feature_names = tfidf.get_feature_names_out()

tfidf_df = pd.DataFrame(
    X_text.toarray(),
    columns=feature_names
)

tfidf_df["root_cause"] = df["root_cause_category"].values

root_cause_word_scores = (
    tfidf_df
    .groupby("root_cause")
    .mean()
)

for root_cause in root_cause_word_scores.index:
    top_words = (
        root_cause_word_scores.loc[root_cause]
        .sort_values(ascending=False)
        .head(10)
    )

    print(f"\n{'='*60}")
    print(root_cause)
    print(f"{'='*60}")
    print(top_words)


Communication Failure
warning      0.049647
shift        0.048155
injury       0.041144
given        0.039939
did          0.038079
area         0.035273
moved        0.033864
hit          0.031438
treatment    0.031204
told         0.031035
Name: Communication Failure, dtype: float64

Design Deficiency
fixed          0.072741
workstation    0.050613
adjusted       0.049389
built          0.047555
height         0.046396
moving         0.040927
layout         0.040418
strain         0.039120
point          0.038912
machine        0.038518
Name: Design Deficiency, dtype: float64

Environmental Condition
heat           0.091322
exhaustion     0.078874
case           0.077244
celsius        0.065341
temperature    0.065341
degrees        0.065341
afternoon      0.061629
area           0.059969
outside        0.056345
42             0.054724
Name: Environmental Condition, dtype: float64

Equipment Failure
respond     0.051349
machine     0.048643
did         0.038267
injury      0.036791


In [39]:
numeric_eda_cols = [
    "risk_score",
    "lost_time_days",
    "description_word_count",
    "description_char_count"
]

correlation_matrix = df[numeric_eda_cols].corr()

display(correlation_matrix.round(2))

,risk_score,lost_time_days,description_word_count,description_char_count
risk_score,1.00,0.49,-0.00,0.01
lost_time_days,0.49,1.00,-0.02,-0.02
description_word_count,-0.00,-0.02,1.00,0.98
description_char_count,0.01,-0.02,0.98,1.00


In [40]:
# Remove temporary EDA-only columns
eda_temp_cols = [
    "incident_year",
    "incident_month",
    "incident_hour",
    "description_word_count",
    "description_char_count"
]

df_clean = df.drop(columns=eda_temp_cols).copy()

print("Final cleaned dataset shape:", df_clean.shape)
print(df_clean.isna().sum().sort_values(ascending=False).head(10))

Final cleaned dataset shape: (8000, 29)
contributing_factor_2     2578
body_part_affected        1135
injury_type               1135
root_cause_description      41
incident_time                0
shift                        0
employee_role                0
equipment_type               0
experience_level             0
equipment_status             0
dtype: int64


In [41]:
df_clean.to_csv(
    "safety_incidents_cleaned.csv",
    index=False
)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.


In [42]:
df = pd.read_csv("safety_incidents_cleaned.csv")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (8000, 29)

Columns:
['incident_date', 'incident_time', 'department', 'location', 'shift', 'employee_role', 'experience_level', 'equipment_type', 'equipment_status', 'incident_type', 'hazard_type', 'incident_description', 'injury_type', 'body_part_affected', 'medical_treatment_required', 'root_cause_category', 'root_cause_description', 'contributing_factor_1', 'contributing_factor_2', 'severity_level', 'risk_level', 'risk_score', 'immediate_action', 'corrective_action', 'preventive_action', 'reported_by_role', 'investigation_status', 'incident_status', 'lost_time_days']


In [43]:
df.head()

,incident_date,incident_time,department,location,shift,employee_role,experience_level,equipment_type,equipment_status,incident_type,...,severity_level,risk_level,risk_score,immediate_action,corrective_action,preventive_action,reported_by_role,investigation_status,incident_status,lost_time_days
0,2021-01-01,1900-01-01 00:09:00,Chemical Processing,Effluent Treatment Plant,Night,Process Technician,1-3 years,Chemical Storage Cabinet,Damaged,Chemical Exposure,...,Serious,High,61.9,"Employee moved to fresh air, contaminated clot...",Review the chemical transfer and handling meth...,"Set up PPE compliance audits, stock availabili...",Shift Manager,Completed,Closed,8
1,2021-01-01,1900-01-01 05:10:00,Assembly,Assembly Line B,Night,Line Supervisor,3-7 years,Pneumatic Tool,Operational,Machine Injury,...,Serious,High,52.6,Machine stopped and isolated; injured employee...,"Inspect the air-powered driver, repair or repl...","Implement permit-to-work audits, lock-out/tag-...",Supervisor,Completed,Closed,17
2,2021-01-01,1900-01-01 07:25:00,Production,Machining Bay,Morning,Production Technician,More than 7 years,Forklift,Operational,Struck By Object,...,Minor,Medium,27.6,Area secured and injured employee sent to the ...,"Secure loose material, install a barrier aroun...",Standardise the shift handover format and use ...,Supervisor,Completed,Closed,0
3,2021-01-01,1900-01-01 08:52:00,Quality Control & Laboratory,Incoming Inspection,Morning,Quality Inspector,3-7 years,Fume Hood,Operational,Chemical Exposure,...,Minor,Medium,27.8,"Employee moved to fresh air, contaminated clot...",Review the chemical transfer and handling meth...,Standardise the shift handover format and use ...,Supervisor,Completed,Closed,0
4,2021-01-01,1900-01-01 09:26:00,Assembly,Sub-assembly Area,Morning,Assembly Operator,More than 7 years,Not Applicable,Not Applicable,Ergonomic Injury,...,Moderate,Low,19.3,Task rotated and injured employee sent for med...,"Adjust the workstation layout, provide adjusta...",Include HSE review at the design and modificat...,Co-worker,Completed,Closed,1


In [44]:
# Features used for initial prediction

feature_cols = [
    "incident_date",
    "incident_time",
    "department",
    "location",
    "shift",
    "employee_role",
    "experience_level",
    "equipment_type",
    "equipment_status",
    "incident_type",
    "hazard_type",
    "incident_description",
    "injury_type",
    "body_part_affected",
    "medical_treatment_required",
    "contributing_factor_1",
    "contributing_factor_2",
    "reported_by_role"
]

# Targets
target_root_cause = "root_cause_category"
target_severity = "severity_level"
target_risk = "risk_level"

print("Number of features:", len(feature_cols))
print("\nFeatures:")
for i, col in enumerate(feature_cols, 1):
    print(f"{i}. {col}")

Number of features: 18

Features:
1. incident_date
2. incident_time
3. department
4. location
5. shift
6. employee_role
7. experience_level
8. equipment_type
9. equipment_status
10. incident_type
11. hazard_type
12. incident_description
13. injury_type
14. body_part_affected
15. medical_treatment_required
16. contributing_factor_1
17. contributing_factor_2
18. reported_by_role


In [45]:
X_root_cause = df[feature_cols].copy()
y_root_cause = df[target_root_cause].copy()

X_severity = df[feature_cols].copy()
y_severity = df[target_severity].copy()

X_risk = df[feature_cols].copy()
y_risk = df[target_risk].copy()

print("Root Cause:", X_root_cause.shape, y_root_cause.shape)
print("Severity:  ", X_severity.shape, y_severity.shape)
print("Risk:      ", X_risk.shape, y_risk.shape)

Root Cause: (8000, 18) (8000,)
Severity:   (8000, 18) (8000,)
Risk:       (8000, 18) (8000,)


In [46]:
# Convert date/time columns
df["incident_date"] = pd.to_datetime(df["incident_date"], errors="coerce")
df["incident_time"] = pd.to_datetime(df["incident_time"], errors="coerce")

# Extract useful temporal features
df["incident_year"] = df["incident_date"].dt.year
df["incident_month"] = df["incident_date"].dt.month
df["incident_dayofweek"] = df["incident_date"].dt.dayofweek

df["incident_hour"] = df["incident_time"].dt.hour
df["incident_minute"] = df["incident_time"].dt.minute

print(df[
    [
        "incident_date",
        "incident_year",
        "incident_month",
        "incident_dayofweek",
        "incident_time",
        "incident_hour",
        "incident_minute"
    ]
].head())

  incident_date  incident_year  incident_month  incident_dayofweek  \
0    2021-01-01           2021               1                   4   
1    2021-01-01           2021               1                   4   
2    2021-01-01           2021               1                   4   
3    2021-01-01           2021               1                   4   
4    2021-01-01           2021               1                   4   

        incident_time  incident_hour  incident_minute  
0 1900-01-01 00:09:00              0                9  
1 1900-01-01 05:10:00              5               10  
2 1900-01-01 07:25:00              7               25  
3 1900-01-01 08:52:00              8               52  
4 1900-01-01 09:26:00              9               26  


In [47]:
feature_cols = [
    "incident_year",
    "incident_month",
    "incident_dayofweek",
    "incident_hour",
    "incident_minute",
    "department",
    "location",
    "shift",
    "employee_role",
    "experience_level",
    "equipment_type",
    "equipment_status",
    "incident_type",
    "hazard_type",
    "incident_description",
    "injury_type",
    "body_part_affected",
    "medical_treatment_required",
    "contributing_factor_1",
    "contributing_factor_2",
    "reported_by_role"
]

X_root_cause = df[feature_cols].copy()
y_root_cause = df["root_cause_category"].copy()

X_severity = df[feature_cols].copy()
y_severity = df["severity_level"].copy()

X_risk = df[feature_cols].copy()
y_risk = df["risk_level"].copy()

print("Features:", len(feature_cols))
print("Root Cause:", X_root_cause.shape)
print("Severity:", X_severity.shape)
print("Risk:", X_risk.shape)

Features: 21
Root Cause: (8000, 21)
Severity: (8000, 21)
Risk: (8000, 21)


In [48]:
from sklearn.model_selection import train_test_split

# Root Cause
X_train_root, X_test_root, y_train_root, y_test_root = train_test_split(
    X_root_cause,
    y_root_cause,
    test_size=0.20,
    random_state=42,
    stratify=y_root_cause
)

# Severity
X_train_severity, X_test_severity, y_train_severity, y_test_severity = train_test_split(
    X_severity,
    y_severity,
    test_size=0.20,
    random_state=42,
    stratify=y_severity
)

# Risk
X_train_risk, X_test_risk, y_train_risk, y_test_risk = train_test_split(
    X_risk,
    y_risk,
    test_size=0.20,
    random_state=42,
    stratify=y_risk
)

print("Root Cause:")
print("Train:", X_train_root.shape)
print("Test: ", X_test_root.shape)

print("\nSeverity:")
print("Train:", X_train_severity.shape)
print("Test: ", X_test_severity.shape)

print("\nRisk:")
print("Train:", X_train_risk.shape)
print("Test: ", X_test_risk.shape)

Root Cause:
Train: (6400, 21)
Test:  (1600, 21)

Severity:
Train: (6400, 21)
Test:  (1600, 21)

Risk:
Train: (6400, 21)
Test:  (1600, 21)


In [49]:
print("ROOT CAUSE")
print("Train:")
print(y_train_root.value_counts(normalize=True).round(3))
print("\nTest:")
print(y_test_root.value_counts(normalize=True).round(3))

print("\nSEVERITY")
print("Train:")
print(y_train_severity.value_counts(normalize=True).round(3))
print("\nTest:")
print(y_test_severity.value_counts(normalize=True).round(3))

print("\nRISK")
print("Train:")
print(y_train_risk.value_counts(normalize=True).round(3))
print("\nTest:")
print(y_test_risk.value_counts(normalize=True).round(3))

ROOT CAUSE
Train:
root_cause_category
Procedure Violation           0.113
Human Error                   0.112
Poor Housekeeping             0.098
Unsafe Working Condition      0.087
Inadequate Training           0.084
Fatigue/Workload              0.071
Poor Maintenance              0.062
Inadequate Risk Assessment    0.061
Supervision Failure           0.056
Design Deficiency             0.052
Equipment Failure             0.048
Communication Failure         0.045
PPE Non-Compliance            0.044
Environmental Condition       0.043
Other                         0.024
Name: proportion, dtype: float64

Test:
root_cause_category
Procedure Violation           0.113
Human Error                   0.112
Poor Housekeeping             0.098
Unsafe Working Condition      0.087
Inadequate Training           0.084
Fatigue/Workload              0.071
Poor Maintenance              0.062
Inadequate Risk Assessment    0.061
Supervision Failure           0.056
Design Deficiency             0.052
Eq

In [50]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

In [51]:
numeric_features = [
    "incident_year",
    "incident_month",
    "incident_dayofweek",
    "incident_hour",
    "incident_minute"
]

categorical_features = [
    "department",
    "location",
    "shift",
    "employee_role",
    "experience_level",
    "equipment_type",
    "equipment_status",
    "incident_type",
    "hazard_type",
    "injury_type",
    "body_part_affected",
    "medical_treatment_required",
    "contributing_factor_1",
    "contributing_factor_2",
    "reported_by_role"
]

text_feature = "incident_description"

print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Text features:", text_feature)

Numerical features: 5
Categorical features: 15
Text features: incident_description


In [52]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=True
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
        ("text", TfidfVectorizer(
            stop_words="english",
            max_features=3000,
            ngram_range=(1, 2),
            min_df=2
        ), text_feature)
    ]
)

In [53]:
X_train_root_processed = preprocessor.fit_transform(X_train_root)
X_test_root_processed = preprocessor.transform(X_test_root)

print("Training matrix shape:", X_train_root_processed.shape)
print("Testing matrix shape:", X_test_root_processed.shape)

Training matrix shape: (6400, 3314)
Testing matrix shape: (1600, 3314)


In [54]:
def create_preprocessor():
    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True
        ))
    ])

    return ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),

            ("cat", categorical_transformer, categorical_features),

            ("text", TfidfVectorizer(
                stop_words="english",
                max_features=3000,
                ngram_range=(1, 2),
                min_df=2
            ), text_feature)
        ]
    )

In [55]:
preprocessor_root = create_preprocessor()
preprocessor_severity = create_preprocessor()
preprocessor_risk = create_preprocessor()

In [56]:
# Root Cause
X_train_root_processed = preprocessor_root.fit_transform(X_train_root)
X_test_root_processed = preprocessor_root.transform(X_test_root)

# Severity
X_train_severity_processed = preprocessor_severity.fit_transform(X_train_severity)
X_test_severity_processed = preprocessor_severity.transform(X_test_severity)

# Risk
X_train_risk_processed = preprocessor_risk.fit_transform(X_train_risk)
X_test_risk_processed = preprocessor_risk.transform(X_test_risk)

print("Root Cause:")
print(X_train_root_processed.shape, X_test_root_processed.shape)

print("\nSeverity:")
print(X_train_severity_processed.shape, X_test_severity_processed.shape)

print("\nRisk:")
print(X_train_risk_processed.shape, X_test_risk_processed.shape)

Root Cause:
(6400, 3314) (1600, 3314)

Severity:
(6400, 3314) (1600, 3314)

Risk:
(6400, 3314) (1600, 3314)


In [57]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.linear_model import SGDClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [58]:
from sklearn.linear_model import LogisticRegression

logistic_root = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    solver="saga",
    random_state=42
)

logistic_severity = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    solver="saga",
    random_state=42
)

logistic_risk = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    solver="saga",
    random_state=42
)

print("Logistic Regression models created successfully.")

Logistic Regression models created successfully.


In [59]:
print("Training Root Cause model...")
logistic_root.fit(
    X_train_root_processed,
    y_train_root
)

print("Training Severity model...")
logistic_severity.fit(
    X_train_severity_processed,
    y_train_severity
)

print("Training Risk model...")
logistic_risk.fit(
    X_train_risk_processed,
    y_train_risk
)

print("\nAll Logistic Regression models trained.")

Training Root Cause model...
Training Severity model...


D:\HCL-Safety-Project\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Training Risk model...

All Logistic Regression models trained.


D:\HCL-Safety-Project\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [60]:
y_pred_root_log = logistic_root.predict(X_test_root_processed)
y_pred_severity_log = logistic_severity.predict(X_test_severity_processed)
y_pred_risk_log = logistic_risk.predict(X_test_risk_processed)

print("Predictions generated successfully.")

Predictions generated successfully.


In [61]:
def evaluate_model(y_true, y_pred, model_name, target_name):
    print("=" * 70)
    print(f"{model_name} — {target_name}")
    print("=" * 70)

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(
        y_true, y_pred,
        average="macro",
        zero_division=0
    )
    recall = recall_score(
        y_true, y_pred,
        average="macro",
        zero_division=0
    )
    f1 = f1_score(
        y_true, y_pred,
        average="macro",
        zero_division=0
    )
    weighted_f1 = f1_score(
        y_true, y_pred,
        average="weighted",
        zero_division=0
    )

    print(f"Accuracy:          {accuracy:.4f}")
    print(f"Macro Precision:   {precision:.4f}")
    print(f"Macro Recall:      {recall:.4f}")
    print(f"Macro F1:          {f1:.4f}")
    print(f"Weighted F1:       {weighted_f1:.4f}")

    print("\nClassification Report:")
    print(
        classification_report(
            y_true,
            y_pred,
            zero_division=0
        )
    )

    return {
        "Model": model_name,
        "Target": target_name,
        "Accuracy": accuracy,
        "Macro Precision": precision,
        "Macro Recall": recall,
        "Macro F1": f1,
        "Weighted F1": weighted_f1
    }

In [62]:
results_logistic = []

results_logistic.append(
    evaluate_model(
        y_test_root,
        y_pred_root_log,
        "Logistic Regression",
        "Root Cause"
    )
)

results_logistic.append(
    evaluate_model(
        y_test_severity,
        y_pred_severity_log,
        "Logistic Regression",
        "Severity"
    )
)

results_logistic.append(
    evaluate_model(
        y_test_risk,
        y_pred_risk_log,
        "Logistic Regression",
        "Risk"
    )
)

Logistic Regression — Root Cause
Accuracy:          0.8044
Macro Precision:   0.7936
Macro Recall:      0.8054
Macro F1:          0.7962
Weighted F1:       0.8058

Classification Report:
                            precision    recall  f1-score   support

     Communication Failure       0.62      0.85      0.71        72
         Design Deficiency       0.73      0.75      0.74        83
   Environmental Condition       0.73      0.91      0.81        70
         Equipment Failure       0.91      0.87      0.89        77
          Fatigue/Workload       0.76      0.86      0.81       113
               Human Error       0.86      0.73      0.79       179
Inadequate Risk Assessment       0.82      0.77      0.79        97
       Inadequate Training       0.86      0.76      0.81       134
                     Other       0.68      0.72      0.70        39
        PPE Non-Compliance       0.77      0.83      0.80        70
         Poor Housekeeping       0.91      0.89      0.90       

In [63]:
import pandas as pd

logistic_results_df = pd.DataFrame(results_logistic)

logistic_results_df

,Model,Target,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1
0,Logistic Regression,Root Cause,0.804375,0.793556,0.805433,0.796163,0.805759
1,Logistic Regression,Severity,0.955000,0.903857,0.943240,0.919422,0.955569
2,Logistic Regression,Risk,0.634375,0.597514,0.643866,0.609505,0.635636


In [64]:
print("ROOT CAUSE — LOGISTIC REGRESSION")
print(classification_report(
    y_test_root,
    y_pred_root_log,
    zero_division=0
))

print("\nSEVERITY — LOGISTIC REGRESSION")
print(classification_report(
    y_test_severity,
    y_pred_severity_log,
    zero_division=0
))

print("\nRISK — LOGISTIC REGRESSION")
print(classification_report(
    y_test_risk,
    y_pred_risk_log,
    zero_division=0
))

ROOT CAUSE — LOGISTIC REGRESSION
                            precision    recall  f1-score   support

     Communication Failure       0.62      0.85      0.71        72
         Design Deficiency       0.73      0.75      0.74        83
   Environmental Condition       0.73      0.91      0.81        70
         Equipment Failure       0.91      0.87      0.89        77
          Fatigue/Workload       0.76      0.86      0.81       113
               Human Error       0.86      0.73      0.79       179
Inadequate Risk Assessment       0.82      0.77      0.79        97
       Inadequate Training       0.86      0.76      0.81       134
                     Other       0.68      0.72      0.70        39
        PPE Non-Compliance       0.77      0.83      0.80        70
         Poor Housekeeping       0.91      0.89      0.90       157
          Poor Maintenance       0.91      0.81      0.86        99
       Procedure Violation       0.80      0.82      0.81       181
       Supervi

demo value testing for logistic reg model

In [65]:
new_incident = pd.DataFrame([{
    "incident_year": 2026,
    "incident_month": 10,
    "incident_dayofweek": 3,
    "incident_hour": 10,
    "incident_minute": 30,

    "department": "Chemical Processing",
    "location": "Chemical Storage Area",
    "shift": "Morning",

    "employee_role": "Process Technician",
    "experience_level": "3-7 years",

    "equipment_type": "Chemical Storage Cabinet",
    "equipment_status": "Operational",

    "incident_type": "Chemical Exposure",
    "hazard_type": "Chemical Substance",

    "incident_description":
        "An operator was transferring solvent in the chemical storage area "
        "when a small leak occurred. The employee was not wearing the "
        "required chemical-resistant gloves and experienced skin irritation "
        "after contact with the substance. The area was isolated and the "
        "employee was sent for medical treatment.",

    "injury_type": "Skin Irritation",
    "body_part_affected": "Hand",

    "medical_treatment_required": "Yes",

    "contributing_factor_1": "PPE Non-Compliance",
    "contributing_factor_2": None,

    "reported_by_role": "Supervisor"
}])

new_incident

,incident_year,incident_month,incident_dayofweek,incident_hour,incident_minute,department,location,shift,employee_role,experience_level,...,equipment_status,incident_type,hazard_type,incident_description,injury_type,body_part_affected,medical_treatment_required,contributing_factor_1,contributing_factor_2,reported_by_role
0,2026,10,3,10,30,Chemical Processing,Chemical Storage Area,Morning,Process Technician,3-7 years,...,Operational,Chemical Exposure,Chemical Substance,An operator was transferring solvent in the ch...,Skin Irritation,Hand,Yes,PPE Non-Compliance,None,Supervisor


In [66]:
new_incident_processed_root = preprocessor_root.transform(new_incident)
new_incident_processed_severity = preprocessor_severity.transform(new_incident)
new_incident_processed_risk = preprocessor_risk.transform(new_incident)

In [67]:
root_prediction = logistic_root.predict(
    new_incident_processed_root
)[0]

severity_prediction = logistic_severity.predict(
    new_incident_processed_severity
)[0]

risk_prediction = logistic_risk.predict(
    new_incident_processed_risk
)[0]

print("Predicted Root Cause :", root_prediction)
print("Predicted Severity   :", severity_prediction)
print("Predicted Risk       :", risk_prediction)

Predicted Root Cause : PPE Non-Compliance
Predicted Severity   : Moderate
Predicted Risk       : Medium


In [68]:
root_probs = logistic_root.predict_proba(
    new_incident_processed_root
)[0]

severity_probs = logistic_severity.predict_proba(
    new_incident_processed_severity
)[0]

risk_probs = logistic_risk.predict_proba(
    new_incident_processed_risk
)[0]

print("ROOT CAUSE PROBABILITIES")
for label, prob in zip(logistic_root.classes_, root_probs):
    print(f"{label}: {prob:.3f}")

print("\nSEVERITY PROBABILITIES")
for label, prob in zip(logistic_severity.classes_, severity_probs):
    print(f"{label}: {prob:.3f}")

print("\nRISK PROBABILITIES")
for label, prob in zip(logistic_risk.classes_, risk_probs):
    print(f"{label}: {prob:.3f}")

ROOT CAUSE PROBABILITIES
Communication Failure: 0.007
Design Deficiency: 0.012
Environmental Condition: 0.004
Equipment Failure: 0.031
Fatigue/Workload: 0.002
Human Error: 0.016
Inadequate Risk Assessment: 0.020
Inadequate Training: 0.051
Other: 0.003
PPE Non-Compliance: 0.748
Poor Housekeeping: 0.013
Poor Maintenance: 0.003
Procedure Violation: 0.019
Supervision Failure: 0.018
Unsafe Working Condition: 0.053

SEVERITY PROBABILITIES
Critical: 0.010
Minor: 0.012
Moderate: 0.920
Serious: 0.043
Severe: 0.015

RISK PROBABILITIES
Critical: 0.034
High: 0.286
Low: 0.037
Medium: 0.643


In [69]:
from sklearn.linear_model import SGDClassifier

sgd_root = SGDClassifier(
    loss="log_loss",
    max_iter=2000,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

sgd_severity = SGDClassifier(
    loss="log_loss",
    max_iter=2000,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

sgd_risk = SGDClassifier(
    loss="log_loss",
    max_iter=2000,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("SGD models created successfully.")

SGD models created successfully.


In [70]:
print("Training SGD Root Cause model...")
sgd_root.fit(
    X_train_root_processed,
    y_train_root
)

print("Training SGD Severity model...")
sgd_severity.fit(
    X_train_severity_processed,
    y_train_severity
)

print("Training SGD Risk model...")
sgd_risk.fit(
    X_train_risk_processed,
    y_train_risk
)

print("\nAll SGD models trained.")

Training SGD Root Cause model...
Training SGD Severity model...
Training SGD Risk model...

All SGD models trained.


In [71]:
y_pred_root_sgd = sgd_root.predict(
    X_test_root_processed
)

y_pred_severity_sgd = sgd_severity.predict(
    X_test_severity_processed
)

y_pred_risk_sgd = sgd_risk.predict(
    X_test_risk_processed
)

print("SGD predictions generated successfully.")

SGD predictions generated successfully.


In [72]:
results_sgd = []

results_sgd.append(
    evaluate_model(
        y_test_root,
        y_pred_root_sgd,
        "SGD Classifier",
        "Root Cause"
    )
)

results_sgd.append(
    evaluate_model(
        y_test_severity,
        y_pred_severity_sgd,
        "SGD Classifier",
        "Severity"
    )
)

results_sgd.append(
    evaluate_model(
        y_test_risk,
        y_pred_risk_sgd,
        "SGD Classifier",
        "Risk"
    )
)

SGD Classifier — Root Cause
Accuracy:          0.8387
Macro Precision:   0.8271
Macro Recall:      0.8312
Macro F1:          0.8267
Weighted F1:       0.8391

Classification Report:
                            precision    recall  f1-score   support

     Communication Failure       0.71      0.92      0.80        72
         Design Deficiency       0.78      0.72      0.75        83
   Environmental Condition       0.72      0.91      0.81        70
         Equipment Failure       0.93      0.92      0.93        77
          Fatigue/Workload       0.86      0.88      0.87       113
               Human Error       0.88      0.87      0.87       179
Inadequate Risk Assessment       0.90      0.81      0.85        97
       Inadequate Training       0.84      0.78      0.81       134
                     Other       0.70      0.72      0.71        39
        PPE Non-Compliance       0.76      0.81      0.79        70
         Poor Housekeeping       0.92      0.90      0.91       157
 

In [73]:
sgd_results_df = pd.DataFrame(results_sgd)

model_comparison = pd.concat(
    [logistic_results_df, sgd_results_df],
    ignore_index=True
)

model_comparison

,Model,Target,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1
0,Logistic Regression,Root Cause,0.804375,0.793556,0.805433,0.796163,0.805759
1,Logistic Regression,Severity,0.955000,0.903857,0.943240,0.919422,0.955569
2,Logistic Regression,Risk,0.634375,0.597514,0.643866,0.609505,0.635636
3,SGD Classifier,Root Cause,0.838750,0.827106,0.831233,0.826674,0.839074
4,SGD Classifier,Severity,0.967500,0.959306,0.943648,0.951187,0.967394
5,SGD Classifier,Risk,0.638750,0.614609,0.613039,0.613743,0.639067


In [74]:
from sklearn.svm import LinearSVC

svm_root = LinearSVC(
    class_weight="balanced",
    max_iter=5000,
    random_state=42
)

svm_severity = LinearSVC(
    class_weight="balanced",
    max_iter=5000,
    random_state=42
)

svm_risk = LinearSVC(
    class_weight="balanced",
    max_iter=5000,
    random_state=42
)

print("Linear SVM models created successfully.")

Linear SVM models created successfully.


In [75]:
print("Training Linear SVM Root Cause model...")
svm_root.fit(
    X_train_root_processed,
    y_train_root
)

print("Training Linear SVM Severity model...")
svm_severity.fit(
    X_train_severity_processed,
    y_train_severity
)

print("Training Linear SVM Risk model...")
svm_risk.fit(
    X_train_risk_processed,
    y_train_risk
)

print("\nAll Linear SVM models trained.")

Training Linear SVM Root Cause model...
Training Linear SVM Severity model...
Training Linear SVM Risk model...

All Linear SVM models trained.


In [76]:
y_pred_root_svm = svm_root.predict(
    X_test_root_processed
)

y_pred_severity_svm = svm_severity.predict(
    X_test_severity_processed
)

y_pred_risk_svm = svm_risk.predict(
    X_test_risk_processed
)

print("Linear SVM predictions generated successfully.")

Linear SVM predictions generated successfully.


In [77]:
results_svm = []

results_svm.append(
    evaluate_model(
        y_test_root,
        y_pred_root_svm,
        "Linear SVM",
        "Root Cause"
    )
)

results_svm.append(
    evaluate_model(
        y_test_severity,
        y_pred_severity_svm,
        "Linear SVM",
        "Severity"
    )
)

results_svm.append(
    evaluate_model(
        y_test_risk,
        y_pred_risk_svm,
        "Linear SVM",
        "Risk"
    )
)

Linear SVM — Root Cause
Accuracy:          0.8781
Macro Precision:   0.8648
Macro Recall:      0.8668
Macro F1:          0.8648
Weighted F1:       0.8783

Classification Report:
                            precision    recall  f1-score   support

     Communication Failure       0.77      0.89      0.83        72
         Design Deficiency       0.92      0.80      0.85        83
   Environmental Condition       0.81      0.90      0.85        70
         Equipment Failure       0.93      0.92      0.93        77
          Fatigue/Workload       0.85      0.88      0.87       113
               Human Error       0.94      0.86      0.90       179
Inadequate Risk Assessment       0.88      0.87      0.88        97
       Inadequate Training       0.88      0.85      0.86       134
                     Other       0.74      0.72      0.73        39
        PPE Non-Compliance       0.83      0.86      0.85        70
         Poor Housekeeping       0.96      0.96      0.96       157
     

In [78]:
svm_results_df = pd.DataFrame(results_svm)

final_model_comparison = pd.concat(
    [
        logistic_results_df,
        sgd_results_df,
        svm_results_df
    ],
    ignore_index=True
)

final_model_comparison

,Model,Target,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1
0,Logistic Regression,Root Cause,0.804375,0.793556,0.805433,0.796163,0.805759
1,Logistic Regression,Severity,0.955000,0.903857,0.943240,0.919422,0.955569
2,Logistic Regression,Risk,0.634375,0.597514,0.643866,0.609505,0.635636
3,SGD Classifier,Root Cause,0.838750,0.827106,0.831233,0.826674,0.839074
4,SGD Classifier,Severity,0.967500,0.959306,0.943648,0.951187,0.967394
5,SGD Classifier,Risk,0.638750,0.614609,0.613039,0.613743,0.639067
6,Linear SVM,Root Cause,0.878125,0.864781,0.866788,0.864761,0.878344
7,Linear SVM,Severity,0.961875,0.949431,0.949137,0.949080,0.961793
8,Linear SVM,Risk,0.638750,0.614495,0.628884,0.621043,0.638427


In [79]:
print("ROOT CAUSE — LINEAR SVM")
print(classification_report(
    y_test_root,
    y_pred_root_svm,
    zero_division=0
))

print("\nSEVERITY — LINEAR SVM")
print(classification_report(
    y_test_severity,
    y_pred_severity_svm,
    zero_division=0
))

print("\nRISK — LINEAR SVM")
print(classification_report(
    y_test_risk,
    y_pred_risk_svm,
    zero_division=0
))

ROOT CAUSE — LINEAR SVM
                            precision    recall  f1-score   support

     Communication Failure       0.77      0.89      0.83        72
         Design Deficiency       0.92      0.80      0.85        83
   Environmental Condition       0.81      0.90      0.85        70
         Equipment Failure       0.93      0.92      0.93        77
          Fatigue/Workload       0.85      0.88      0.87       113
               Human Error       0.94      0.86      0.90       179
Inadequate Risk Assessment       0.88      0.87      0.88        97
       Inadequate Training       0.88      0.85      0.86       134
                     Other       0.74      0.72      0.73        39
        PPE Non-Compliance       0.83      0.86      0.85        70
         Poor Housekeeping       0.96      0.96      0.96       157
          Poor Maintenance       0.88      0.87      0.87        99
       Procedure Violation       0.87      0.93      0.90       181
       Supervision Fail

In [80]:

# PHASE 1: PREPROCESSING AUDIT

import pandas as pd

print("=" * 60)
print("1. DATASET")
print("=" * 60)
print("Shape:", df.shape)
print("Feature count:", len(feature_cols))
print("Feature columns unique:", len(feature_cols) == len(set(feature_cols)))

missing_columns = [c for c in feature_cols if c not in df.columns]
print("Missing feature columns:", missing_columns)

print("\n" + "=" * 60)
print("2. TARGET VALIDATION")
print("=" * 60)

target_cols = [
    "root_cause_category",
    "severity_level",
    "risk_level"
]

for col in target_cols:
    print(
        f"{col}: exists={col in df.columns}, "
        f"missing={df[col].isna().sum() if col in df.columns else 'N/A'}"
    )

print("\n" + "=" * 60)
print("3. FEATURE MISSING VALUES")
print("=" * 60)

print(
    df[feature_cols]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .loc[lambda s: s > 0]
)

print("\n" + "=" * 60)
print("4. TEXT VALIDATION")
print("=" * 60)

text = df["incident_description"].fillna("").astype(str)

print("Missing descriptions:", df["incident_description"].isna().sum())
print("Empty descriptions:", text.str.strip().eq("").sum())
print("Unique descriptions:", text.nunique())
print("Sample description:\n", text.iloc[0])

print("\n" + "=" * 60)
print("5. TRAIN/TEST SPLIT")
print("=" * 60)

for name, X_train, X_test, y_train, y_test in [
    ("Root Cause", X_train_root, X_test_root, y_train_root, y_test_root),
    ("Severity", X_train_severity, X_test_severity,
     y_train_severity, y_test_severity),
    ("Risk", X_train_risk, X_test_risk, y_train_risk, y_test_risk),
]:
    print(f"\n{name}")
    print("Train:", X_train.shape, y_train.shape)
    print("Test: ", X_test.shape, y_test.shape)
    print("Index overlap:", len(set(X_train.index) & set(X_test.index)))
    print("Feature columns match:", list(X_train.columns) == feature_cols)

print("\n" + "=" * 60)
print("6. POSSIBLE LEAKAGE / POST-INCIDENT FEATURES")
print("=" * 60)

review_cols = [
    "injury_type",
    "body_part_affected",
    "medical_treatment_required",
    "contributing_factor_1",
    "contributing_factor_2",
]

for col in review_cols:
    print(f"{col}: included={col in feature_cols}")

print("\nAudit completed. No data or models were modified.")


1. DATASET
Shape: (8000, 34)
Feature count: 21
Feature columns unique: True
Missing feature columns: []

2. TARGET VALIDATION
root_cause_category: exists=True, missing=0
severity_level: exists=True, missing=0
risk_level: exists=True, missing=0

3. FEATURE MISSING VALUES
contributing_factor_2    2578
body_part_affected       1135
injury_type              1135
dtype: int64

4. TEXT VALIDATION
Missing descriptions: 0
Empty descriptions: 0
Unique descriptions: 8000
Sample description:
 A process technician came into contact with caustic solution during sampling at the chemical storage cabinet. She was not wearing the required chemical-resistant gloves. She was referred to a specialist for an inhalation injury and could not perform normal duties for weeks.

5. TRAIN/TEST SPLIT

Root Cause
Train: (6400, 21) (6400,)
Test:  (1600, 21) (1600,)
Index overlap: 0
Feature columns match: True

Severity
Train: (6400, 21) (6400,)
Test:  (1600, 21) (1600,)
Index overlap: 0
Feature columns match: True



In [81]:

# PHASE 2: CHECK FEATURE AVAILABILITY AND POSSIBLE LEAKAGE

targets = [
    "root_cause_category",
    "severity_level",
    "risk_level"
]

print("FEATURE AVAILABILITY AUDIT")
print("=" * 65)

for col in feature_cols:
    print(f"\n{col}")
    print("  Missing:", df[col].isna().sum())
    print("  Unique values:", df[col].nunique(dropna=True))
    print("  Examples:", df[col].dropna().astype(str).unique()[:5])

print("\n" + "=" * 65)
print("TARGET-RELATED COLUMNS NOT CURRENTLY USED AS FEATURES")

excluded_cols = [
    "root_cause_description",
    "risk_score",
    "lost_time_days",
    "corrective_action",
    "preventive_action",
    "immediate_action",
    "investigation_status",
    "incident_status"
]

for col in excluded_cols:
    if col in df.columns:
        print(f"{col}: available, excluded from feature_cols = {col not in feature_cols}")

print("\n" + "=" * 65)
print("CONTRIBUTING FACTORS VS ROOT CAUSE")

for col in ["contributing_factor_1", "contributing_factor_2"]:
    if col in df.columns:
        print(f"\n{col}")
        print(pd.crosstab(
            df[col].fillna("Missing"),
            df["root_cause_category"],
            normalize="index"
        ).round(2).head(10))

print("\nAudit complete. No columns or models were modified.")


FEATURE AVAILABILITY AUDIT

incident_year
  Missing: 0
  Unique values: 6
  Examples: <StringArray>
['2021', '2022', '2023', '2024', '2025']
Length: 5, dtype: str

incident_month
  Missing: 0
  Unique values: 12
  Examples: <StringArray>
['1', '2', '3', '4', '5']
Length: 5, dtype: str

incident_dayofweek
  Missing: 0
  Unique values: 7
  Examples: <StringArray>
['4', '5', '6', '0', '1']
Length: 5, dtype: str

incident_hour
  Missing: 0
  Unique values: 24
  Examples: <StringArray>
['0', '5', '7', '8', '9']
Length: 5, dtype: str

incident_minute
  Missing: 0
  Unique values: 60
  Examples: <StringArray>
['9', '10', '25', '52', '26']
Length: 5, dtype: str

department
  Missing: 0
  Unique values: 11
  Examples: <StringArray>
[         'Chemical Processing',                     'Assembly',
                   'Production', 'Quality Control & Laboratory',
        'Warehouse & Logistics']
Length: 5, dtype: str

location
  Missing: 0
  Unique values: 54
  Examples: <StringArray>
['Effluent Tr

In [82]:
# STEP 1: Explicit prediction-time feature policy

CORE_FEATURES = [
    "incident_year",
    "incident_month",
    "incident_dayofweek",
    "incident_hour",
    "incident_minute",
    "department",
    "location",
    "shift",
    "employee_role",
    "experience_level",
    "equipment_type",
    "equipment_status",
    "incident_type",
    "hazard_type",
    "incident_description",
    "reported_by_role",
]

OPTIONAL_FEATURES = [
    "contributing_factor_1",
    "contributing_factor_2",
]

HELD_OUT_FEATURES = [
    "injury_type",
    "body_part_affected",
    "medical_treatment_required",
    "risk_score",
    "lost_time_days",
    "root_cause_description",
    "immediate_action",
    "corrective_action",
    "preventive_action",
    "investigation_status",
    "incident_status",
]

TARGET_COLUMNS = [
    "root_cause_category",
    "severity_level",
    "risk_level",
]

PROPOSED_FEATURES = CORE_FEATURES + OPTIONAL_FEATURES

print("CORE FEATURES:", len(CORE_FEATURES))
print("OPTIONAL FEATURES:", len(OPTIONAL_FEATURES))
print("PROPOSED TOTAL:", len(PROPOSED_FEATURES))

print("\n--- Schema checks ---")
print("Duplicate proposed features:",
      len(PROPOSED_FEATURES) != len(set(PROPOSED_FEATURES)))

print("Missing from cleaned dataframe:",
      sorted(set(PROPOSED_FEATURES + TARGET_COLUMNS) - set(df.columns)))

print("Missing from X_train:",
      sorted(set(PROPOSED_FEATURES) - set(X_train.columns)))

print("Missing from X_test:",
      sorted(set(PROPOSED_FEATURES) - set(X_test.columns)))

print("\n--- Optional feature missingness ---")
print(
    df[OPTIONAL_FEATURES]
    .isna()
    .sum()
    .to_dict()
)

print("\n--- Held-out fields present in proposed schema ---")
print(sorted(set(PROPOSED_FEATURES) & set(HELD_OUT_FEATURES)))

print("\n--- Existing versus proposed schema ---")
print("Existing feature count:", len(feature_cols))
print("Proposed feature count:", len(PROPOSED_FEATURES))
print("Features being removed:",
      sorted(set(feature_cols) - set(PROPOSED_FEATURES)))
print("Features being added:",
      sorted(set(PROPOSED_FEATURES) - set(feature_cols)))

CORE FEATURES: 16
OPTIONAL FEATURES: 2
PROPOSED TOTAL: 18

--- Schema checks ---
Duplicate proposed features: False
Missing from cleaned dataframe: []
Missing from X_train: []
Missing from X_test: []

--- Optional feature missingness ---
{'contributing_factor_1': 0, 'contributing_factor_2': 2578}

--- Held-out fields present in proposed schema ---
[]

--- Existing versus proposed schema ---
Existing feature count: 21
Proposed feature count: 18
Features being removed: ['body_part_affected', 'injury_type', 'medical_treatment_required']
Features being added: []


In [83]:
# PHASE 1: LOCK PREDICTION-TIME FEATURES
# Existing cells/models are preserved.

feature_cols = [
    "incident_year",
    "incident_month",
    "incident_dayofweek",
    "incident_hour",
    "incident_minute",
    "department",
    "location",
    "shift",
    "employee_role",
    "experience_level",
    "equipment_type",
    "equipment_status",
    "incident_type",
    "hazard_type",
    "incident_description",
    "reported_by_role",
]

target_cols = [
    "root_cause_category",
    "severity_level",
    "risk_level",
]

# Validate schema
assert len(feature_cols) == len(set(feature_cols))
assert all(col in df.columns for col in feature_cols + target_cols)

# Rebuild datasets
X = df[feature_cols].copy()

X_root_cause = X.copy()
y_root_cause = df["root_cause_category"].copy()

X_severity = X.copy()
y_severity = df["severity_level"].copy()

X_risk = X.copy()
y_risk = df["risk_level"].copy()

print("Feature count:", len(feature_cols))
print("Feature columns:", feature_cols)
print("Root cause:", X_root_cause.shape, y_root_cause.shape)
print("Severity:", X_severity.shape, y_severity.shape)
print("Risk:", X_risk.shape, y_risk.shape)
print("\nMissing feature values:")
print(X[feature_cols].isna().sum().loc[lambda s: s > 0])

Feature count: 16
Feature columns: ['incident_year', 'incident_month', 'incident_dayofweek', 'incident_hour', 'incident_minute', 'department', 'location', 'shift', 'employee_role', 'experience_level', 'equipment_type', 'equipment_status', 'incident_type', 'hazard_type', 'incident_description', 'reported_by_role']
Root cause: (8000, 16) (8000,)
Severity: (8000, 16) (8000,)
Risk: (8000, 16) (8000,)

Missing feature values:
Series([], dtype: int64)


In [84]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

numeric_features = [
    "incident_year",
    "incident_month",
    "incident_dayofweek",
    "incident_hour",
    "incident_minute",
]

categorical_features = [
    "department",
    "location",
    "shift",
    "employee_role",
    "experience_level",
    "equipment_type",
    "equipment_status",
    "incident_type",
    "hazard_type",
    "reported_by_role",
]

text_feature = "incident_description"

def create_preprocessor():
    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]),
                numeric_features,
            ),
            (
                "categorical",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]),
                categorical_features,
            ),
            (
                "text",
                TfidfVectorizer(max_features=3000, ngram_range=(1, 2)),
                text_feature,
            ),
        ],
        remainder="drop",
    )

preprocessors = {
    "root_cause_category": create_preprocessor(),
    "severity_level": create_preprocessor(),
    "risk_level": create_preprocessor(),
}

print("Preprocessors created:", list(preprocessors.keys()))
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Text features:", 1)

Preprocessors created: ['root_cause_category', 'severity_level', 'risk_level']
Numeric features: 5
Categorical features: 10
Text features: 1


In [85]:
from sklearn.model_selection import train_test_split

datasets = {
    "root_cause_category": (X_root_cause, y_root_cause),
    "severity_level": (X_severity, y_severity),
    "risk_level": (X_risk, y_risk),
}

splits = {}

for target, (X_target, y_target) in datasets.items():
    X_train, X_test, y_train, y_test = train_test_split(
        X_target,
        y_target,
        test_size=0.2,
        random_state=42,
        stratify=y_target,
    )

    splits[target] = (X_train, X_test, y_train, y_test)

    print(
        f"{target}: "
        f"train={X_train.shape}, test={X_test.shape}"
    )

root_cause_category: train=(6400, 16), test=(1600, 16)
severity_level: train=(6400, 16), test=(1600, 16)
risk_level: train=(6400, 16), test=(1600, 16)


In [86]:
X_train_transformed = {}
X_test_transformed = {}

for target, (X_train, X_test, y_train, y_test) in splits.items():
    preprocessor = preprocessors[target]

    X_train_transformed[target] = preprocessor.fit_transform(X_train)
    X_test_transformed[target] = preprocessor.transform(X_test)

    print(
        f"{target}: "
        f"train={X_train_transformed[target].shape}, "
        f"test={X_test_transformed[target].shape}"
    )

root_cause_category: train=(6400, 3239), test=(1600, 3239)
severity_level: train=(6400, 3239), test=(1600, 3239)
risk_level: train=(6400, 3239), test=(1600, 3239)


In [87]:
from importlib.util import find_spec

libraries = ["xgboost", "catboost", "lightgbm"]

for library in libraries:
    print(f"{library}: {'Installed' if find_spec(library) else 'Not installed'}")

xgboost: Installed
catboost: Installed
lightgbm: Installed


In [88]:
import time
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import accuracy_score, f1_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.base import clone

results = []
trained_models = {}
validation_preprocessors = {}

for target, (X_train, X_test, y_train, y_test) in splits.items():

    # Split training data into train and validation
    X_sub, X_val, y_sub, y_val = train_test_split(
        X_train, y_train,
        test_size=0.2,
        random_state=42,
        stratify=y_train
    )

    # Fit preprocessing only on the smaller training subset
    prep = create_preprocessor()
    X_sub_prep = prep.fit_transform(X_sub)
    X_val_prep = prep.transform(X_val)

    # ComplementNB needs non-negative numeric features
    nb_prep = ColumnTransformer([
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", MinMaxScaler())
        ]), numeric_features),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features),
        ("text", TfidfVectorizer(max_features=3000, ngram_range=(1, 2)),
         text_feature)
    ])

    X_sub_nb = nb_prep.fit_transform(X_sub)
    X_val_nb = nb_prep.transform(X_val)

    models = {
        "Logistic Regression": (
            LogisticRegression(max_iter=2000),
            X_sub_prep, X_val_prep
        ),
        "Linear SVM": (
            LinearSVC(),
            X_sub_prep, X_val_prep
        ),
        "SGD Classifier": (
            SGDClassifier(random_state=42, max_iter=2000),
            X_sub_prep, X_val_prep
        ),
        "Complement Naive Bayes": (
            ComplementNB(),
            X_sub_nb, X_val_nb
        ),
    }

    for model_name, (model, Xtr, Xv) in models.items():
        start = time.time()
        model.fit(Xtr, y_sub)
        predictions = model.predict(Xv)

        results.append({
            "Target": target,
            "Model": model_name,
            "Accuracy": accuracy_score(y_val, predictions),
            "Macro F1": f1_score(y_val, predictions, average="macro"),
            "Training Time (s)": round(time.time() - start, 2)
        })

        trained_models[(target, model_name)] = model

    validation_preprocessors[target] = prep

results_df = pd.DataFrame(results).sort_values(
    ["Target", "Macro F1"], ascending=[True, False]
)

display(results_df.round(4))

,Target,Model,Accuracy,Macro F1,Training Time (s)
9,risk_level,Linear SVM,0.6320,0.5966,0.68
8,risk_level,Logistic Regression,0.6484,0.5934,0.56
10,risk_level,SGD Classifier,0.6047,0.5550,0.23
11,risk_level,Complement Naive Bayes,0.5430,0.4727,0.01
1,root_cause_category,Linear SVM,0.8133,0.8025,1.22
2,root_cause_category,SGD Classifier,0.7836,0.7678,0.32
0,root_cause_category,Logistic Regression,0.7320,0.7012,0.90
3,root_cause_category,Complement Naive Bayes,0.6469,0.6042,0.02
5,severity_level,Linear SVM,0.9375,0.9238,0.37
6,severity_level,SGD Classifier,0.9352,0.9108,0.14


In [89]:
from sklearn.base import clone

final_models = {}
final_preprocessors = {}

for target, (X_train, X_test, y_train, y_test) in splits.items():
    preprocessor = create_preprocessor()

    X_train_processed = preprocessor.fit_transform(X_train)

    model = LinearSVC()
    model.fit(X_train_processed, y_train)

    final_preprocessors[target] = preprocessor
    final_models[target] = model

    print(f"{target}: trained on {X_train.shape[0]} rows")

root_cause_category: trained on 6400 rows
severity_level: trained on 6400 rows
risk_level: trained on 6400 rows


In [90]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

test_results = []

for target, (X_train, X_test, y_train, y_test) in splits.items():
    preprocessor = final_preprocessors[target]
    model = final_models[target]

    X_test_processed = preprocessor.transform(X_test)
    predictions = model.predict(X_test_processed)

    accuracy = accuracy_score(y_test, predictions)
    macro_f1 = f1_score(y_test, predictions, average="macro")

    test_results.append({
        "Target": target,
        "Accuracy": accuracy,
        "Macro F1": macro_f1,
    })

    print(f"\n{'=' * 45}\nTARGET: {target}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Macro F1: {macro_f1:.4f}")
    print(classification_report(y_test, predictions, zero_division=0))

test_results_df = pd.DataFrame(test_results)
display(test_results_df.round(4))


TARGET: root_cause_category
Accuracy: 0.8306
Macro F1: 0.8190
                            precision    recall  f1-score   support

     Communication Failure       0.79      0.88      0.83        72
         Design Deficiency       0.87      0.80      0.83        83
   Environmental Condition       0.85      0.89      0.87        70
         Equipment Failure       0.87      0.79      0.83        77
          Fatigue/Workload       0.82      0.87      0.84       113
               Human Error       0.90      0.85      0.88       179
Inadequate Risk Assessment       0.78      0.85      0.81        97
       Inadequate Training       0.72      0.75      0.73       134
                     Other       0.84      0.54      0.66        39
        PPE Non-Compliance       0.85      0.89      0.87        70
         Poor Housekeeping       0.91      0.92      0.92       157
          Poor Maintenance       0.88      0.85      0.86        99
       Procedure Violation       0.80      0.89     

,Target,Accuracy,Macro F1
0,root_cause_category,0.8306,0.8190
1,severity_level,0.9388,0.9221
2,risk_level,0.6406,0.6214


analysing risk level to see why result coming so poor

In [91]:
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

target = "risk_level"

X_train, X_test, y_train, y_test = splits[target]

X_test_processed = final_preprocessors[target].transform(X_test)
risk_predictions = final_models[target].predict(X_test_processed)

print("CLASSIFICATION REPORT")
print(classification_report(
    y_test,
    risk_predictions,
    zero_division=0
))

print("CONFUSION MATRIX")
labels = final_models[target].classes_

display(pd.DataFrame(
    confusion_matrix(y_test, risk_predictions, labels=labels),
    index=[f"Actual {label}" for label in labels],
    columns=[f"Predicted {label}" for label in labels]
))

CLASSIFICATION REPORT
              precision    recall  f1-score   support

    Critical       0.66      0.46      0.54       112
        High       0.60      0.60      0.60       422
         Low       0.70      0.68      0.69       402
      Medium       0.63      0.67      0.65       664

    accuracy                           0.64      1600
   macro avg       0.65      0.60      0.62      1600
weighted avg       0.64      0.64      0.64      1600

CONFUSION MATRIX


,Predicted Critical,Predicted High,Predicted Low,Predicted Medium
Actual Critical,52,52,0,8
Actual High,26,252,10,134
Actual Low,0,3,273,126
Actual Medium,1,110,105,448


In [92]:
print("Risk-level distribution:")
print(df["risk_level"].value_counts())

print("\nRisk score availability:")
if "risk_score" in df.columns:
    print(df["risk_score"].describe())
    print("\nMean risk score by risk level:")
    print(df.groupby("risk_level")["risk_score"].agg(
        ["count", "mean", "median", "min", "max"]
    ))
else:
    print("risk_score column not found.")

Risk-level distribution:
risk_level
Medium      3323
High        2109
Low         2009
Critical     559
Name: count, dtype: int64

Risk score availability:
count    8000.000000
mean       41.610238
std        20.654510
min         1.900000
25%        25.000000
50%        39.500000
75%        56.300000
max       100.000000
Name: risk_score, dtype: float64

Mean risk score by risk level:
            count       mean  median   min    max
risk_level                                       
Critical      559  83.092308    81.8  75.1  100.0
High         2109  61.037364    60.4  50.1   75.0
Low          2009  16.733051    17.4   1.9   25.0
Medium       3323  37.342401    37.0  25.1   50.0


In [93]:
import numpy as np
import pandas as pd

df["risk_level_from_score"] = pd.cut(
    df["risk_score"],
    bins=[-np.inf, 25, 50, 75, np.inf],
    labels=["Low", "Medium", "High", "Critical"]
)

print("Agreement with existing risk_level:")
print(
    (df["risk_level"].astype(str) ==
     df["risk_level_from_score"].astype(str)).mean()
)

print("\nMismatched rows:")
display(
    df.loc[
        df["risk_level"].astype(str) !=
        df["risk_level_from_score"].astype(str),
        ["risk_score", "risk_level", "risk_level_from_score"]
    ].head(10)
)

Agreement with existing risk_level:
1.0

Mismatched rows:


,risk_score,risk_level,risk_level_from_score


In [94]:
# Check how risk levels vary across categorical features

categorical_check = [
    "department",
    "shift",
    "employee_role",
    "experience_level",
    "equipment_type",
    "equipment_status",
    "incident_type",
    "hazard_type",
    "location",
]

for col in categorical_check:
    print(f"\n--- {col} ---")
    display(pd.crosstab(
        df[col],
        df["risk_level"],
        normalize="index"
    ).round(2))


--- department ---


risk_level,Critical,High,Low,Medium
department,,,,
Assembly,0.04,0.21,0.32,0.43
Chemical Processing,0.06,0.23,0.22,0.49
Electrical,0.10,0.43,0.15,0.32
Fabrication & Welding,0.08,0.29,0.23,0.41
Facilities & Housekeeping,0.02,0.17,0.39,0.41
Maintenance,0.08,0.30,0.22,0.41
Packaging,0.04,0.18,0.35,0.42
Production,0.08,0.26,0.25,0.41
Quality Control & Laboratory,0.06,0.23,0.26,0.45



--- shift ---


risk_level,Critical,High,Low,Medium
shift,,,,
Evening,0.07,0.26,0.24,0.43
Morning,0.07,0.27,0.25,0.41
Night,0.07,0.26,0.27,0.40



--- employee_role ---


risk_level,Critical,High,Low,Medium
employee_role,,,,
Assembly Operator,0.05,0.22,0.29,0.44
Boiler Operator,0.06,0.23,0.26,0.45
CNC Operator,0.07,0.20,0.30,0.44
Chemical Operator,0.06,0.23,0.23,0.48
Chemist,0.00,0.52,0.17,0.31
Contractor,0.05,0.32,0.22,0.41
Driver,0.06,0.19,0.22,0.53
Electrical Supervisor,0.04,0.38,0.17,0.42
Electrical Technician,0.09,0.41,0.16,0.34


... (8 outputs left out)


--- hazard_type ---


risk_level,Critical,High,Low,Medium
hazard_type,,,,
Chemical Substance,0.04,0.27,0.11,0.58
Confined Space,0.14,0.51,0.01,0.33
Electrical Energy,0.16,0.52,0.01,0.31
Excessive Heat,0.02,0.19,0.23,0.56
Falling Object,0.11,0.36,0.05,0.48
Fire Hazard,0.24,0.48,0.01,0.27
Gas/Pressure,0.25,0.49,0.00,0.27
Height Work,0.10,0.41,0.05,0.45
Inadequate PPE,0.01,0.25,0.17,0.57



--- location ---


risk_level,Critical,High,Low,Medium
location,,,,
Admin Block,0.03,0.13,0.33,0.50
Assembly Line A,0.04,0.17,0.34,0.46
Assembly Line B,0.06,0.26,0.31,0.38
Blending Area,0.09,0.28,0.22,0.41
Boiler House,0.15,0.26,0.20,0.39
Cable Gallery,0.06,0.40,0.13,0.41
Canteen,0.00,0.19,0.35,0.46
Chemical Store,0.05,0.25,0.19,0.51
Compressor Room,0.05,0.28,0.32,0.35


In [95]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = splits["risk_level"]

X_sub, X_val, y_sub, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

prep = create_preprocessor()
X_sub_p = prep.fit_transform(X_sub)
X_val_p = prep.transform(X_val)

for name, weights in [
    ("Baseline", None),
    ("Balanced", "balanced"),
]:
    model = LinearSVC(class_weight=weights)
    model.fit(X_sub_p, y_sub)
    pred = model.predict(X_val_p)

    print(f"\n--- {name} ---")
    print("Accuracy:", round(accuracy_score(y_val, pred), 4))
    print("Macro F1:", round(f1_score(y_val, pred, average="macro"), 4))
    print(classification_report(
        y_val, pred, zero_division=0
    ))


--- Baseline ---
Accuracy: 0.632
Macro F1: 0.5966
              precision    recall  f1-score   support

    Critical       0.61      0.37      0.46        89
        High       0.61      0.59      0.60       337
         Low       0.68      0.68      0.68       322
      Medium       0.62      0.67      0.65       532

    accuracy                           0.63      1280
   macro avg       0.63      0.58      0.60      1280
weighted avg       0.63      0.63      0.63      1280


--- Balanced ---
Accuracy: 0.6219
Macro F1: 0.601
              precision    recall  f1-score   support

    Critical       0.54      0.47      0.50        89
        High       0.60      0.60      0.60       337
         Low       0.66      0.74      0.70       322
      Medium       0.63      0.59      0.61       532

    accuracy                           0.62      1280
   macro avg       0.60      0.60      0.60      1280
weighted avg       0.62      0.62      0.62      1280



back to model training

In [96]:
for target, (X_train, X_test, y_train, y_test) in splits.items():
    print(
        f"{target}: "
        f"train={X_train.shape}, "
        f"test={X_test.shape}, "
        f"features={X_train.shape[1]}"
    )

print("\nExisting final SVM models:", list(final_models.keys()))
print("Existing preprocessors:", list(final_preprocessors.keys()))

root_cause_category: train=(6400, 16), test=(1600, 16), features=16
severity_level: train=(6400, 16), test=(1600, 16), features=16
risk_level: train=(6400, 16), test=(1600, 16), features=16

Existing final SVM models: ['root_cause_category', 'severity_level', 'risk_level']
Existing preprocessors: ['root_cause_category', 'severity_level', 'risk_level']


In [97]:
import time
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

tree_results = []

for target, (X_train, X_test, y_train, y_test) in splits.items():
    X_sub, X_val, y_sub, y_val = train_test_split(
        X_train, y_train,
        test_size=0.2,
        random_state=42,
        stratify=y_train
    )

    prep = create_preprocessor()
    X_sub_p = prep.fit_transform(X_sub)
    X_val_p = prep.transform(X_val)

    models = {
        "Random Forest": RandomForestClassifier(
            n_estimators=150,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ),
        "Extra Trees": ExtraTreesClassifier(
            n_estimators=150,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ),
    }

    for name, model in models.items():
        start = time.time()
        model.fit(X_sub_p, y_sub)
        pred = model.predict(X_val_p)

        tree_results.append({
            "Target": target,
            "Model": name,
            "Accuracy": accuracy_score(y_val, pred),
            "Macro F1": f1_score(y_val, pred, average="macro"),
            "Training Time (s)": round(time.time() - start, 2)
        })

tree_results_df = pd.DataFrame(tree_results)

comparison_results_df = pd.concat(
    [results_df, tree_results_df],
    ignore_index=True
).sort_values(
    ["Target", "Macro F1"],
    ascending=[True, False]
)

display(comparison_results_df.round(4))

,Target,Model,Accuracy,Macro F1,Training Time (s)
17,risk_level,Extra Trees,0.6820,0.6694,0.92
16,risk_level,Random Forest,0.6516,0.6510,0.68
0,risk_level,Linear SVM,0.6320,0.5966,0.68
1,risk_level,Logistic Regression,0.6484,0.5934,0.56
2,risk_level,SGD Classifier,0.6047,0.5550,0.23
3,risk_level,Complement Naive Bayes,0.5430,0.4727,0.01
4,root_cause_category,Linear SVM,0.8133,0.8025,1.22
13,root_cause_category,Extra Trees,0.7945,0.7900,0.97
12,root_cause_category,Random Forest,0.7914,0.7869,0.83
5,root_cause_category,SGD Classifier,0.7836,0.7678,0.32


In [98]:
import time
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

structured_features = numeric_features + categorical_features

boost_results = []

for target, (X_train, X_test, y_train, y_test) in splits.items():
    X_sub, X_val, y_sub, y_val = train_test_split(
        X_train, y_train,
        test_size=0.2,
        random_state=42,
        stratify=y_train
    )

    # Structured features only; no risk_score or text description
    structured_prep = ColumnTransformer([
        ("numeric", SimpleImputer(strategy="median"), numeric_features),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1
            ))
        ]), categorical_features)
    ])

    X_sub_p = structured_prep.fit_transform(X_sub)
    X_val_p = structured_prep.transform(X_val)

    model = HistGradientBoostingClassifier(
        max_iter=150,
        learning_rate=0.1,
        random_state=42
    )

    start = time.time()
    model.fit(X_sub_p, y_sub)
    pred = model.predict(X_val_p)

    boost_results.append({
        "Target": target,
        "Model": "HistGradientBoosting (structured)",
        "Accuracy": accuracy_score(y_val, pred),
        "Macro F1": f1_score(y_val, pred, average="macro"),
        "Training Time (s)": round(time.time() - start, 2)
    })

boost_results_df = pd.DataFrame(boost_results)

comparison_results_df = pd.concat(
    [comparison_results_df, boost_results_df],
    ignore_index=True
).sort_values(
    ["Target", "Macro F1"],
    ascending=[True, False]
)

display(comparison_results_df.round(4))

,Target,Model,Accuracy,Macro F1,Training Time (s)
0,risk_level,Extra Trees,0.6820,0.6694,0.92
1,risk_level,Random Forest,0.6516,0.6510,0.68
2,risk_level,Linear SVM,0.6320,0.5966,0.68
3,risk_level,Logistic Regression,0.6484,0.5934,0.56
4,risk_level,SGD Classifier,0.6047,0.5550,0.23
5,risk_level,Complement Naive Bayes,0.5430,0.4727,0.01
20,risk_level,HistGradientBoosting (structured),0.4719,0.4039,0.99
6,root_cause_category,Linear SVM,0.8133,0.8025,1.22
7,root_cause_category,Extra Trees,0.7945,0.7900,0.97
8,root_cause_category,Random Forest,0.7914,0.7869,0.83


In [99]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import f1_score
import pandas as pd

tuning_results = []

for target, (X_train, X_test, y_train, y_test) in splits.items():
    X_sub, X_val, y_sub, y_val = train_test_split(
        X_train, y_train,
        test_size=0.2,
        random_state=42,
        stratify=y_train
    )

    prep = create_preprocessor()
    X_sub_p = prep.fit_transform(X_sub)
    X_val_p = prep.transform(X_val)

    candidates = {
        "Linear SVM": (
            LinearSVC(),
            {"C": [0.5, 1.0, 2.0]}
        ),
        "Extra Trees": (
            ExtraTreesClassifier(
                random_state=42,
                n_jobs=-1,
                class_weight="balanced"
            ),
            {
                "n_estimators": [150, 300],
                "max_features": ["sqrt", 0.5],
                "min_samples_leaf": [1, 2]
            }
        )
    }

    for name, (model, params) in candidates.items():
        search = GridSearchCV(
            model,
            params,
            scoring="f1_macro",
            cv=StratifiedKFold(
                n_splits=3, shuffle=True, random_state=42
            ),
            n_jobs=-1
        )

        search.fit(X_sub_p, y_sub)
        pred = search.best_estimator_.predict(X_val_p)

        tuning_results.append({
            "Target": target,
            "Model": name,
            "Validation Macro F1": f1_score(
                y_val, pred, average="macro"
            ),
            "Best parameters": search.best_params_
        })

display(
    pd.DataFrame(tuning_results)
    .sort_values(["Target", "Validation Macro F1"],
                 ascending=[True, False])
)

,Target,Model,Validation Macro F1,Best parameters
5,risk_level,Extra Trees,0.669358,"{'max_features': 'sqrt', 'min_samples_leaf': 1..."
4,risk_level,Linear SVM,0.596590,{'C': 1.0}
0,root_cause_category,Linear SVM,0.812962,{'C': 0.5}
1,root_cause_category,Extra Trees,0.797544,"{'max_features': 'sqrt', 'min_samples_leaf': 2..."
3,severity_level,Extra Trees,0.942515,"{'max_features': 'sqrt', 'min_samples_leaf': 1..."
2,severity_level,Linear SVM,0.920357,{'C': 0.5}


In [100]:
%pip install xgboost catboost lightgbm

Note: you may need to restart the kernel to use updated packages.


In [101]:
import xgboost
import catboost
import lightgbm

print("XGBoost:", xgboost.__version__)
print("CatBoost:", catboost.__version__)
print("LightGBM:", lightgbm.__version__)

XGBoost: 3.4.1
CatBoost: 1.2.10
LightGBM: 4.7.0


In [102]:
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

boosting_results = []

for target, (X_train, _, y_train, _) in splits.items():
    X_sub, X_val, y_sub, y_val = train_test_split(
        X_train,
        y_train,
        test_size=0.2,
        random_state=42,
        stratify=y_train
    )

    preprocessor = create_preprocessor()
    X_sub_t = preprocessor.fit_transform(X_sub, y_sub)
    X_val_t = preprocessor.transform(X_val)

    encoder = LabelEncoder()
    y_sub_enc = encoder.fit_transform(y_sub)
    y_val_enc = encoder.transform(y_val)

    models = {
        "XGBoost": XGBClassifier(
            n_estimators=250,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="multi:softprob",
            num_class=len(encoder.classes_),
            eval_metric="mlogloss",
            tree_method="hist",
            n_jobs=-1,
            random_state=42
        ),
        "LightGBM": LGBMClassifier(
            n_estimators=250,
            learning_rate=0.05,
            num_leaves=31,
            verbosity=-1,
            n_jobs=-1,
            random_state=42
        ),
        "CatBoost": CatBoostClassifier(
            iterations=250,
            depth=6,
            learning_rate=0.05,
            loss_function="MultiClass",
            verbose=False,
            random_seed=42,
            thread_count=-1
        )
    }

    for model_name, model in models.items():
        start = time.time()

        # CatBoost receives a dense matrix; the other two can use sparse input.
        if model_name == "CatBoost":
            X_fit = X_sub_t.toarray().astype(np.float32)
            X_valid = X_val_t.toarray().astype(np.float32)
        else:
            X_fit = X_sub_t
            X_valid = X_val_t

        model.fit(X_fit, y_sub_enc)
        predictions = model.predict(X_valid).astype(int).ravel()

        boosting_results.append({
            "target": target,
            "model": model_name,
            "accuracy": accuracy_score(y_val_enc, predictions),
            "macro_f1": f1_score(
                y_val_enc, predictions, average="macro"
            ),
            "training_seconds": round(time.time() - start, 1)
        })

        print(f"Completed: {target} — {model_name}")

boosting_results_df = pd.DataFrame(boosting_results).sort_values(
    ["target", "macro_f1"], ascending=[True, False]
)

display(boosting_results_df)

Completed: root_cause_category — XGBoost
Completed: root_cause_category — LightGBM
Completed: root_cause_category — CatBoost
Completed: severity_level — XGBoost
Completed: severity_level — LightGBM
Completed: severity_level — CatBoost
Completed: risk_level — XGBoost
Completed: risk_level — LightGBM
Completed: risk_level — CatBoost


,target,model,accuracy,macro_f1,training_seconds
6,risk_level,XGBoost,0.688281,0.655595,34.2
7,risk_level,LightGBM,0.671875,0.642826,9.0
8,risk_level,CatBoost,0.647656,0.584163,48.0
1,root_cause_category,LightGBM,0.840625,0.838311,32.4
0,root_cause_category,XGBoost,0.826562,0.822673,100.8
2,root_cause_category,CatBoost,0.759375,0.762021,270.6
4,severity_level,LightGBM,0.942969,0.946579,10.2
3,severity_level,XGBoost,0.941406,0.937986,28.9
5,severity_level,CatBoost,0.935937,0.918267,60.9


In [103]:
# Combine results from all completed model-comparison rounds.
all_results = pd.concat(
    [
        comparison_results_df,
        boosting_results_df
    ],
    ignore_index=True,
    sort=False
)

# Remove duplicate model-target entries if any.
all_results = all_results.drop_duplicates(
    subset=["target", "model"],
    keep="last"
)

# Rank every model within each target by validation Macro F1.
all_results = all_results.sort_values(
    ["target", "macro_f1"],
    ascending=[True, False]
)

display(all_results)

# Show the current winner for each target.
overall_winners = all_results.loc[
    all_results.groupby("target")["macro_f1"].idxmax()
].sort_values("target")

display(overall_winners)

,Target,Model,Accuracy,Macro F1,Training Time (s),target,model,accuracy,macro_f1,training_seconds
21,NaN,NaN,NaN,NaN,NaN,risk_level,XGBoost,0.688281,0.655595,34.2
22,NaN,NaN,NaN,NaN,NaN,risk_level,LightGBM,0.671875,0.642826,9.0
23,NaN,NaN,NaN,NaN,NaN,risk_level,CatBoost,0.647656,0.584163,48.0
24,NaN,NaN,NaN,NaN,NaN,root_cause_category,LightGBM,0.840625,0.838311,32.4
25,NaN,NaN,NaN,NaN,NaN,root_cause_category,XGBoost,0.826562,0.822673,100.8
26,NaN,NaN,NaN,NaN,NaN,root_cause_category,CatBoost,0.759375,0.762021,270.6
27,NaN,NaN,NaN,NaN,NaN,severity_level,LightGBM,0.942969,0.946579,10.2
28,NaN,NaN,NaN,NaN,NaN,severity_level,XGBoost,0.941406,0.937986,28.9
29,NaN,NaN,NaN,NaN,NaN,severity_level,CatBoost,0.935937,0.918267,60.9
20,severity_level,HistGradientBoosting (structured),0.4125,0.288764,1.21,NaN,NaN,NaN,NaN,NaN


,Target,Model,Accuracy,Macro F1,Training Time (s),target,model,accuracy,macro_f1,training_seconds
21,NaN,NaN,NaN,NaN,NaN,risk_level,XGBoost,0.688281,0.655595,34.2
24,NaN,NaN,NaN,NaN,NaN,root_cause_category,LightGBM,0.840625,0.838311,32.4
27,NaN,NaN,NaN,NaN,NaN,severity_level,LightGBM,0.942969,0.946579,10.2


In [104]:
# Standardize the original model comparison columns
base_results = comparison_results_df.rename(columns={
    "Target": "target",
    "Model": "model",
    "Accuracy": "accuracy",
    "Macro F1": "macro_f1",
    "Training Time (s)": "training_seconds"
})

# Standardize boosting results
boost_results = boosting_results_df.copy()
boost_results.columns = boost_results.columns.str.strip().str.lower()

# Combine all available results
all_results = pd.concat(
    [base_results, boost_results],
    ignore_index=True
)

# Keep only valid result rows and remove duplicate models
all_results = all_results.dropna(
    subset=["target", "model", "macro_f1"]
).drop_duplicates(
    subset=["target", "model"],
    keep="last"
)

# Rank models within each target
all_results = all_results.sort_values(
    ["target", "macro_f1"],
    ascending=[True, False]
)

display(all_results)

# Best model for each target
overall_winners = all_results.loc[
    all_results.groupby("target")["macro_f1"].idxmax()
].sort_values("target")

display(overall_winners)

,target,model,accuracy,macro_f1,training_seconds
0,risk_level,Extra Trees,0.682031,0.669358,0.92
21,risk_level,XGBoost,0.688281,0.655595,34.20
1,risk_level,Random Forest,0.651563,0.650966,0.68
22,risk_level,LightGBM,0.671875,0.642826,9.00
2,risk_level,Linear SVM,0.632031,0.596590,0.68
3,risk_level,Logistic Regression,0.648438,0.593365,0.56
23,risk_level,CatBoost,0.647656,0.584163,48.00
4,risk_level,SGD Classifier,0.604688,0.555044,0.23
5,risk_level,Complement Naive Bayes,0.542969,0.472688,0.01
6,risk_level,HistGradientBoosting (structured),0.471875,0.403950,0.99


,target,model,accuracy,macro_f1,training_seconds
0,risk_level,Extra Trees,0.682031,0.669358,0.92
24,root_cause_category,LightGBM,0.840625,0.838311,32.40
27,severity_level,LightGBM,0.942969,0.946579,10.20


In [108]:
import time
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.ensemble import ExtraTreesClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, f1_score

tuning_configs = {
    "risk_level": {
        "model": ExtraTreesClassifier(
            class_weight="balanced",
            random_state=42,
            n_jobs=1
        ),
        "params": {
            "model__n_estimators": [200, 300],
            "model__max_features": ["sqrt", 0.7],
            "model__min_samples_leaf": [1, 2]
        }
    },
    "root_cause_category": {
        "model": LGBMClassifier(
            verbosity=-1,
            random_state=42,
            n_jobs=1
        ),
        "params": {
            "model__n_estimators": [150, 250, 350],
            "model__learning_rate": [0.03, 0.05, 0.1],
            "model__num_leaves": [15, 31, 50]
        }
    },
    "severity_level": {
        "model": LGBMClassifier(
            verbosity=-1,
            random_state=42,
            n_jobs=1
        ),
        "params": {
            "model__n_estimators": [150, 250, 350],
            "model__learning_rate": [0.03, 0.05, 0.1],
            "model__num_leaves": [15, 31, 50]
        }
    }
}

tuned_models = {}
tuned_preprocessors = {}
tuning_summary = []

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

for target, config in tuning_configs.items():
    X_train, _, y_train, _ = splits[target]

    X_sub, X_val, y_sub, y_val = train_test_split(
        X_train,
        y_train,
        test_size=0.2,
        random_state=42,
        stratify=y_train
    )

    pipeline = Pipeline([
        ("preprocessor", create_preprocessor()),
        ("model", config["model"])
    ])

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=config["params"],
        scoring="f1_macro",
        cv=cv,
        n_jobs=-1,
        refit=True,
        verbose=1
    )

    start = time.time()
    search.fit(X_sub, y_sub)

    predictions = search.best_estimator_.predict(X_val)

    tuned_models[target] = search.best_estimator_.named_steps["model"]
    tuned_preprocessors[target] = search.best_estimator_.named_steps["preprocessor"]

    tuning_summary.append({
        "target": target,
        "best_cv_macro_f1": search.best_score_,
        "validation_accuracy": accuracy_score(y_val, predictions),
        "validation_macro_f1": f1_score(
            y_val, predictions, average="macro"
        ),
        "best_parameters": search.best_params_,
        "tuning_seconds": round(time.time() - start, 1)
    })

    print(f"\nCompleted tuning: {target}")

tuning_summary_df = pd.DataFrame(tuning_summary)
display(tuning_summary_df)

Fitting 3 folds for each of 8 candidates, totalling 24 fits

Completed tuning: risk_level
Fitting 3 folds for each of 27 candidates, totalling 81 fits

Completed tuning: root_cause_category
Fitting 3 folds for each of 27 candidates, totalling 81 fits

Completed tuning: severity_level


,target,best_cv_macro_f1,validation_accuracy,validation_macro_f1,best_parameters,tuning_seconds
0,risk_level,0.648482,0.667188,0.656969,"{'model__max_features': 'sqrt', 'model__min_sa...",230.1
1,root_cause_category,0.811673,0.838281,0.835754,"{'model__learning_rate': 0.05, 'model__n_estim...",1264.0
2,severity_level,0.925949,0.943750,0.946662,"{'model__learning_rate': 0.1, 'model__n_estima...",323.0


In [109]:
import time
import pandas as pd

from sklearn.ensemble import ExtraTreesClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

selected_models = {
    "risk_level": ExtraTreesClassifier(
        n_estimators=150,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    "root_cause_category": LGBMClassifier(
        n_estimators=250,
        learning_rate=0.05,
        num_leaves=31,
        verbosity=-1,
        random_state=42,
        n_jobs=-1
    ),
    "severity_level": LGBMClassifier(
        n_estimators=250,
        learning_rate=0.05,
        num_leaves=31,
        verbosity=-1,
        random_state=42,
        n_jobs=-1
    )
}

final_models = {}
final_preprocessors = {}
final_test_results = []

for target, (X_train, X_test, y_train, y_test) in splits.items():
    preprocessor = create_preprocessor()

    X_train_processed = preprocessor.fit_transform(X_train)
    X_test_processed = preprocessor.transform(X_test)

    model = selected_models[target]

    start = time.time()
    model.fit(X_train_processed, y_train)
    training_seconds = round(time.time() - start, 2)

    predictions = model.predict(X_test_processed)

    final_models[target] = model
    final_preprocessors[target] = preprocessor

    final_test_results.append({
        "target": target,
        "model": type(model).__name__,
        "accuracy": accuracy_score(y_test, predictions),
        "macro_f1": f1_score(y_test, predictions, average="macro"),
        "training_seconds": training_seconds
    })

    print(f"\n{'=' * 60}")
    print(f"TARGET: {target}")
    print(classification_report(y_test, predictions, zero_division=0))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, predictions))

final_test_results_df = pd.DataFrame(final_test_results)
display(final_test_results_df)


TARGET: root_cause_category
                            precision    recall  f1-score   support

     Communication Failure       0.88      0.83      0.86        72
         Design Deficiency       0.93      0.82      0.87        83
   Environmental Condition       0.88      0.87      0.88        70
         Equipment Failure       0.93      0.86      0.89        77
          Fatigue/Workload       0.85      0.89      0.87       113
               Human Error       0.87      0.87      0.87       179
Inadequate Risk Assessment       0.83      0.85      0.84        97
       Inadequate Training       0.75      0.85      0.80       134
                     Other       0.97      0.72      0.82        39
        PPE Non-Compliance       0.91      0.87      0.89        70
         Poor Housekeeping       0.88      0.93      0.90       157
          Poor Maintenance       0.93      0.87      0.90        99
       Procedure Violation       0.82      0.89      0.85       181
       Supervision

,target,model,accuracy,macro_f1,training_seconds
0,root_cause_category,LGBMClassifier,0.852500,0.852123,49.10
1,severity_level,LGBMClassifier,0.936250,0.930396,14.92
2,risk_level,ExtraTreesClassifier,0.674375,0.660914,1.25


In [110]:
from pathlib import Path
import joblib

model_dir = Path("saved_models")
model_dir.mkdir(exist_ok=True)

for target in final_models:
    artifact = {
        "model": final_models[target],
        "preprocessor": final_preprocessors[target],
        "target": target
    }

    path = model_dir / f"{target}_pipeline.joblib"
    joblib.dump(artifact, path)
    print(f"Saved: {path}")

print("\nAll model pipelines saved.")

Saved: saved_models\root_cause_category_pipeline.joblib
Saved: saved_models\severity_level_pipeline.joblib
Saved: saved_models\risk_level_pipeline.joblib

All model pipelines saved.


In [111]:
from pathlib import Path
import joblib

model_dir = Path("saved_models")

targets = [
    "root_cause_category",
    "severity_level",
    "risk_level"
]

artifacts = {}

for target in targets:
    path = model_dir / f"{target}_pipeline.joblib"

    if not path.exists():
        raise FileNotFoundError(f"Model file not found: {path.resolve()}")

    artifacts[target] = joblib.load(path)

print("Loaded prediction pipelines:")
for target, artifact in artifacts.items():
    print(f"{target}: {type(artifact['model']).__name__}")

Loaded prediction pipelines:
root_cause_category: LGBMClassifier
severity_level: LGBMClassifier
risk_level: ExtraTreesClassifier


In [112]:
import pandas as pd

feature_cols = [
    "incident_year",
    "incident_month",
    "incident_dayofweek",
    "incident_hour",
    "incident_minute",
    "department",
    "location",
    "shift",
    "employee_role",
    "experience_level",
    "equipment_type",
    "equipment_status",
    "incident_type",
    "hazard_type",
    "incident_description",
    "reported_by_role"
]

def predict_incident(incident):
    missing = [col for col in feature_cols if col not in incident]

    if missing:
        raise ValueError(f"Missing required input fields: {missing}")

    input_df = pd.DataFrame(
        [{col: incident[col] for col in feature_cols}],
        columns=feature_cols
    )

    predictions = {}

    for target, artifact in artifacts.items():
        processed = artifact["preprocessor"].transform(input_df)
        predictions[target] = artifact["model"].predict(processed)[0]

    return predictions

In [113]:
sample_incident = {
    "incident_year": 2026,
    "incident_month": 10,
    "incident_dayofweek": 0,
    "incident_hour": 10,
    "incident_minute": 30,
    "department": "Production",
    "location": "Production Line 1",
    "shift": "Morning",
    "employee_role": "Production Technician",
    "experience_level": "1-3 years",
    "equipment_type": "Forklift",
    "equipment_status": "Operational",
    "incident_type": "Struck By Object",
    "hazard_type": "Moving Machinery",
    "incident_description": (
        "A worker was struck by a moving forklift near the production line."
    ),
    "reported_by_role": "Supervisor"
}

prediction = predict_incident(sample_incident)

print("Predicted root cause:", prediction["root_cause_category"])
print("Predicted severity:", prediction["severity_level"])
print("Predicted risk level:", prediction["risk_level"])

Predicted root cause: Communication Failure
Predicted severity: Serious
Predicted risk level: High


In [118]:
sample_incident = {
    "incident_year": 2032,
    "incident_month": 18,
    "incident_dayofweek": 8,
    "incident_hour": 0,
    "incident_minute": 70,
    "department": "Supply",
    "location": "Chain line",
    "shift": "Night",
    "employee_role": "Employee",
    "experience_level": "10 years",
    "equipment_type": "machinery and everything",
    "equipment_status": "not working and non existing",
    "incident_type": "cut in pieces and fire everywhere",
    "hazard_type": "Moving chain and fire",
    "incident_description": (
        "A worker got cut in pieces on a moving chain machinery and the whole factory caught fire."
    ),
    "reported_by_role": "Manager"
}

prediction = predict_incident(sample_incident)

print("Predicted root cause:", prediction["root_cause_category"])
print("Predicted severity:", prediction["severity_level"])
print("Predicted risk level:", prediction["risk_level"])

Predicted root cause: Procedure Violation
Predicted severity: Moderate
Predicted risk level: High


In [119]:
%pip install fastapi uvicorn

  Using cached uvicorn-0.54.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
Using cached uvicorn-0.54.0-py3-none-any.whl (87 kB)
Using cached click-8.5.0-py3-none-any.whl (125 kB)
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------- ----------- 1.6/2.2 MB 8.0 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 6.8 MB/s  0:00:00

   ------------ ---------------------------  3/10 [click]
   ------------------------ ---------------  6/10 [uvicorn]
   ---------------------------- -----------  7/10 [starlette]
   -------------------------------- -------  8/10 [pydantic]
   -------------------------------- -------  8/10 [pydantic]
   ------------------------------------ ---  9/10 [fastapi]
   ---------------------------------------- 10/10 [fastapi]

Note: you may need to restart the kernel to use updated packages.


In [120]:
assert set(artifacts.keys()) == {
    "root_cause_category",
    "severity_level",
    "risk_level"
}

test_prediction = predict_incident(sample_incident)

assert set(test_prediction.keys()) == {
    "root_cause_category",
    "severity_level",
    "risk_level"
}

print("Prediction pipeline verified.")
print(test_prediction)

Prediction pipeline verified.
{'root_cause_category': 'Procedure Violation', 'severity_level': 'Moderate', 'risk_level': 'High'}


In [121]:
%%writefile app.py
from pathlib import Path

import joblib
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, ConfigDict

BASE_DIR = Path(__file__).resolve().parent
MODEL_DIR = BASE_DIR / "saved_models"

targets = [
    "root_cause_category",
    "severity_level",
    "risk_level",
]

feature_cols = [
    "incident_year", "incident_month", "incident_dayofweek",
    "incident_hour", "incident_minute", "department", "location",
    "shift", "employee_role", "experience_level", "equipment_type",
    "equipment_status", "incident_type", "hazard_type",
    "incident_description", "reported_by_role",
]

artifacts = {
    target: joblib.load(MODEL_DIR / f"{target}_pipeline.joblib")
    for target in targets
}

app = FastAPI(title="Safety Incident Prediction API")


class Incident(BaseModel):
    model_config = ConfigDict(extra="forbid")

    incident_year: int
    incident_month: int
    incident_dayofweek: int
    incident_hour: int
    incident_minute: int
    department: str
    location: str
    shift: str
    employee_role: str
    experience_level: str
    equipment_type: str
    equipment_status: str
    incident_type: str
    hazard_type: str
    incident_description: str
    reported_by_role: str


@app.get("/health")
def health():
    return {"status": "ok", "models_loaded": list(artifacts.keys())}


@app.post("/predict")
def predict(incident: Incident):
    try:
        input_df = pd.DataFrame(
            [incident.model_dump()],
            columns=feature_cols,
        )

        predictions = {}
        for target, artifact in artifacts.items():
            processed = artifact["preprocessor"].transform(input_df)
            predictions[target] = str(
                artifact["model"].predict(processed)[0]
            )

        return {"predictions": predictions}

    except Exception as exc:
        raise HTTPException(
            status_code=500,
            detail=f"Prediction failed: {exc}",
        ) from exc

Writing app.py


In [122]:
from pathlib import Path

print("Working directory:", Path.cwd())
print("API file exists:", Path("app.py").exists())
print("Saved models exist:", Path("saved_models").exists())

Working directory: D:\HCL-Safety-Project
API file exists: True
Saved models exist: True


In [3]:
categorical_cols = [
    "department",
    "location",
    "shift",
    "employee_role",
    "experience_level",
    "equipment_type",
    "equipment_status",
    "incident_type",
    "hazard_type",
    "reported_by_role",
]

for col in categorical_cols:
    print(f"\n--- {col} ---")
    print(sorted(df[col].dropna().astype(str).unique().tolist()))


--- department ---
['Assembly', 'Chemical Processing', 'Electrical', 'Fabrication & Welding', 'Facilities & Housekeeping', 'Maintenance', 'Packaging', 'Production', 'Quality Control & Laboratory', 'Utilities', 'Warehouse & Logistics']

--- location ---
['Admin Block', 'Assembly Line A', 'Assembly Line B', 'Blending Area', 'Boiler House', 'Cable Gallery', 'Canteen', 'Chemical Store', 'Compressor Room', 'Cooling Tower Area', 'Corridors and Stairways', 'Cutting Area', 'DG Room', 'Despatch Area', 'Dispatch Packing Line', 'Effluent Treatment Plant', 'Electrical Room', 'Fabrication Shop', 'Final Inspection Bay', 'Finished Goods Store', 'Heavy Fabrication Yard', 'Incoming Inspection', 'Labelling Section', 'Loading Dock', 'MCC Room', 'Machining Bay', 'Main Warehouse', 'Maintenance Workshop', 'Motor Repair Bay', 'Moulding Area', 'Packing Hall', 'Paint Booth Area', 'Palletizing Area', 'Parking Area', 'Plant Floor', 'Press Shop', 'Process Control Area', 'Production Line 1', 'Production Line 2', 